# DASC512 Assignment 2 - Dental Caries Prediction

# Building on Orelnko et al. (2025), NHANES 2017-2018

# Student ID: 201949015

In [1]:
#===============================================================================
# CELL 0: CODE QUALITY SETUP
#===============================================================================
# A linter is a tool that reads your code and flags problems BEFORE you run
# it. Think of it as a spell-checker for Python. It catches undefined
# variables, unused imports, and formatting inconsistencies without executing
# a single line. This is standard professional practice.
#
# Flake8 — the industry standard Python linter. It checks:
#   1. Indentation errors
#   2. Whitespace errors
#   3. Blank line conventions
#   4. Statement errors (e.g. bare except, comparison to None)
#   5. Imported but unused modules (F401)
#   6. Redefinition of unused name from import (F811)
#   7. Local variable assigned but never used (F841)
#   8. Trailing whitespace (W291)
#===============================================================================

import subprocess

# Install Flake8 and pycodestyle
print("Installing Flake8 and pycodestyle...")
subprocess.run(
    ["pip", "install", "flake8", "pycodestyle", "--quiet"],
    check=True,
)

# Confirm installation by printing the version
version_result = subprocess.run(
    ["flake8", "--version"],
    capture_output=True,
    text=True,
    check=True,
)
print(f"Flake8 version: {version_result.stdout.strip()}")

print("\nLinting tools ready.")
print("Command used:")
print(
    "  flake8 <notebook.py> --max-line-length=100 "
    "--ignore=E241,E265,E302,E305,E402,E501,F401,W291,W503"
)

Installing Flake8 and pycodestyle...
Flake8 version: 7.3.0 (mccabe: 0.7.0, pycodestyle: 2.14.0, pyflakes: 3.4.0) CPython 3.12.13 on
Linux

Linting tools ready.
Command used:
  flake8 <notebook.py> --max-line-length=100 --ignore=E241,E265,E302,E305,E402,E501,F401,W291,W503


In [2]:
#===============================================================================
# CHAPTER 1: CONFIGURATION AND REPRODUCIBILITY
#===============================================================================
# WHAT IS REPRODUCIBILITY AND WHY DOES IT MATTER CLINICALLY?
# Machine learning involves randomness in multiple places: which patients
# end up in which fold, how a Random Forest samples its training data, how
# a neural network initialises its weights, which neurons Dropout removes
# each pass. If we do not fix these random processes to a known starting point
# (a "seed"), two runs of identical code produce different numbers.
# In clinical research, a result that changes every time you run the code
# is not a result - it is noise. Setting SEED = 42 everywhere ensures that
# every number in this notebook is byte-for-byte reproducible.
#===============================================================================

import os
import random
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# sklearn imports
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_curve,)

# PyTorch imports - deep learning framework
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# IterativeImputer prints convergence warnings that add no information.
# We suppress only that specific warning - nothing else - so genuine errors
# still surface
warnings.filterwarnings(
    "ignore",
    message=r"\[IterativeImputer\]")

#===============================================================================
# GOOGLE DRIVE MOUNTING
#===============================================================================
# Colab's runtime resets between sessions. Moutning Drive persists the dataset
# and outputs across sessions so we never lose figures or results.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully.")
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - using local directory.")

#===============================================================================
# GLOBAL RANDOM SEED
#===============================================================================
# Why 42? Convention only - the number itself is arbitrary. What matters is
# that it is fixed, documented, and applied to every library that has its
# own random state.
SEED: int = 42

def set_all_seeds(seed: int = SEED) -> None:
    """Seed every radnom number generator used in this notebook.

    Machine learning uses randomness in data splitting (StratifiedKFold),
    bootstrap sampling (Random Forest)m weight initialisation (ANN),
    dropout masking, and batch shuffling (DataLoader). Without fixing all
    of these, results change between runs. This function seeds Python's
    built-in random module, Numpy, and all PyTorch generators in one call.

    Args:
        seed: Integer seed applied to all random number generators.
              Default is the global SEED contant (42).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        # Seed all GPUs if multiple are available
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

#===============================================================================
# FILE PATHS
#===============================================================================
# All paths are derived from PROJECT_DIR.
if IN_COLAB:
    PROJECT_DIR = "/content/drive/MyDrive/DASC512_Assessment2"
else:
    PROJECT_DIR = os.path.abspath("DASC512_Assessment2_local")

DATA_PATH = os.path.join(PROJECT_DIR, "all_2018_processed_cleaned_data.txt")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")

# Create output directories if they do not exist
for _dir in (OUTPUT_DIR, FIGURE_DIR, TABLE_DIR):
    os.makedirs(_dir, exist_ok=True)

print(f"Project directory : {PROJECT_DIR}")
print(f"Dataset expected  : {DATA_PATH}")
print(f"Figures will save : {FIGURE_DIR}")

#===============================================================================
# EXPERIMENT CONTROL FLAGS
#===============================================================================
# QUICK_MODE = True: fast smoke test run to check pipeline validity
# QUICK_MODE = False: full submission run for final submission
QUICK_MODE: bool= False
RUN_MULTICLASS: bool = True

#===============================================================================
# CROSS-VALIDATION CONFIGURATION
#===============================================================================
# Stratified K-Fold cross-validation with k=5 used. Cross-validation is
# the standard method for evaluating machine learning models when data is
# limited. Rather than a single train/test split (which is sensitive to
# which patients happen to fall in each group), we repeat the evaluation
# five times on non-overlapping folds and average the results.
#
# STRATIFIED means each fold preserves the class ratio of the full dataset.
# With only ~14% of children having caries, an unstratified split could put
# all caries-positive children into one fold by chance, producing misleading
# results. Stratification prevents this.
#
# WHY K=5 SPECIFICALLY?
# K=5 is the curriculum standard, and commonly used value in the
# clinical ML literature. It gives 80% training / 20% validation per fold -
# sufficient data to train while keeping validation sets large enough to
# estimate performance reliabily.
#
# References for cross-validation methdology:
# Kohavi R (1995). A study of cross-validation and bootstrap for accuracy
# estimation and model selection. IJCAI.
# James, Witten, Hastie, and Tibshirani. An Introduction to Statistical
# Learning
N_SPLITS: int=5

#===============================================================================
# HYPERPARAMETERS
#===============================================================================
if QUICK_MODE:
    # Reduced settings for smoke testing - confirms the pipeline runs
    # end-to-end without committing to the full computational cost.
    IMPUTER_MAX_ITER: int = 2             # Iterative imputation passes
    IMPUTER_NEAREST_FEATURES: int  = 20   # Features used per imputation model
    ANN_EPOCHS: int = 20                # Training epochs for the neural network
    RF_N_ESTIMATORS: int = 60           # Number of trees in the Random Forest
else:
    # Full settigns for submission run
    IMPUTER_MAX_ITER = 5
    IMPUTER_NEAREST_FEATURES = 30
    ANN_EPOCHS = 100                   # Matches Week 6 practical exactly
    RF_N_ESTIMATORS = 200

# ANN architecture
# WHY THESE VALUES?
# batch_size=32: Standard value. Small enough to introduce beneficial
# gradient noise (regularisation effect), large enough for BatchNorm to
# estimate stable mean/variance statistics.
# learning_rate=0.001: Default Adam learning rate - well-validated starting
# point for tabular health data (Kingma & Ba (2015).
# weight_decay=1e-4: L2 regularisation. Penalises large weights, reducing
# overfittig. Critical here: 440 input features with small strate
# (Children n=921) creates geniune overfitting risk.
# hidden_layers=(128, 64): Two hidden layesr of decreasing width. This
# progressive compression forces the network to learn abstract
# representations rather than memorising training patients.
# dropout=0.30: Randomly zeros 30% of activations each forward pass during
# training. Forces the network to learn redundant representations - no
# single neuron can dominate. Srivastava et al. (2014)
ANN_BATCH_SIZE: int = 32
ANN_LEARNING_RATE: float = 1e-3
ANN_WEIGHT_DECAY: float = 1e-4
ANN_HIDDEN_LAYERS: tuple = (128, 64)
ANN_DROPOUT: float = 0.30

# Random Forest - class_weight="balanced" is non-negotiable.
# WHY: The children stratum has only ~14% caries prevalence. An unweighted
# forest would maximise accuracy by predicting "no caries" for everyone -
# achieving 86% accuracy while being clinically useless. "balanced"
# upweights minority class errors so the model is penalised for missing
# caries-positive children, not rewarding them for ignoring them.
RF_CLASS_WEIGHT: str = "balanced"

#===============================================================================
# NHANES VARIABLE LABEL DICTIONARY
#===============================================================================
# NHANES variable use coded names (e.g., LBXBPB) that are meaningless to
# anyone outside epidemiology. If the labels are raw codes, the clinical
# interpretation is invisible.
#
# This dictionary maps codes to readbale clinical labels. Variables not in
# this dictionary fall back to their raw code name (handled in Chapter 11).
NHANES_LABELS: dict = {
    # Demographic
    "RIDAGEYR":   "Age (years)",
    "RIAGENDR":   "Sex (1=Male, 2=Female)",
    "INDFMPIR":   "Income-to-poverty ratio",
    "DMDEDUC2":   "Education level (adult)",
    "DMDFMSIZ":   "Family size",
    # Blood metals — Orlenko's key finding: lead as socioeconomic proxy
    "LBXBPB":     "Blood lead (µg/dL)",
    "LBXBCD":     "Blood cadmium (µg/L)",
    "LBXTHG":     "Blood mercury (µg/L)",
    "LBXBSE":     "Blood selenium (µg/L)",
    # Haematology — prominent in children's cluster (Orlenko Fig 3D)
    "LBXWBCSI":   "White blood cell count (SI)",
    "LBXHGB":     "Haemoglobin (g/dL)",
    "LBXHCT":     "Haematocrit (%)",
    "LBXMCVSI":   "Mean corpuscular volume (fL)",
    "LBXRDW":     "Red cell distribution width (%)",
    "LBXPLTSI":   "Platelet count (SI)",
    "LBXIRN":     "Serum iron (µg/dL)",
    "LBDFERSI":   "Serum ferritin (ng/mL)",
    "LBXRBCSI":   "Red blood cell count (SI)",
    # Biochemistry
    "LBXSCR":     "Serum creatinine (mg/dL)",
    "LBXSGL":     "Serum glucose (mg/dL)",
    "LBXSCA":     "Serum calcium (mg/dL)",
    "LBXSTP":     "Serum total protein (g/dL)",
    "LBXSCH":     "Serum cholesterol (mg/dL)",
    "LBXSTB":     "Serum total bilirubin (mg/dL)",
    "LBXSIR":     "Serum iron (µg/dL)",
    "LBXGH":      "Glycohaemoglobin HbA1c (%)",
    "LBXHSCRP":   "High-sensitivity CRP (mg/L)",
    # Smoking — cotinine is the biomarker for tobacco smoke exposure
    "LBXCOT":     "Serum cotinine — smoking exposure (ng/mL)",
    # Dietary — sugar is the primary cariogenic dietary factor
    "DR2TSUGR":   "Total sugar intake (g/day)",
    "DR2TKCAL":   "Total energy intake (kcal/day)",
    "DR2TCARB":   "Total carbohydrate intake (g/day)",
    "DR2TFIBE":   "Dietary fibre intake (g/day)",
    "DR2TPROT":   "Protein intake (g/day)",
    "DR2TCALC":   "Calcium intake (mg/day)",
    "DR2TIRON":   "Iron intake (mg/day)",
    "DR2TVD":     "Vitamin D intake (µg/day)",
    "DR2TVC":     "Vitamin C intake (mg/day)",
    # Physical examination
    "BMXBMI":     "BMI (kg/m²)",
    "BMXWT":      "Weight (kg)",
    "BMXHT":      "Height (cm)",
    "BPXSY1":     "Systolic BP — reading 1 (mmHg)",
    "BPXDI1":     "Diastolic BP — reading 1 (mmHg)",
    "BPXPLS":     "Pulse rate (bpm)",
    # Sleep
    "SLD012":     "Sleep duration weekdays (hrs)",
    "SLD013":     "Sleep duration weekends (hrs)",
    # Oral health questionnaire
    "OHQ030":     "Self-rated oral health",
    "OHQ620":     "Dental visits in past year",
    # Insurance
    "HIQ011":     "Health insurance coverage",}

#===============================================================================
# VISUAL STYLING - consistent, publication quality
#===============================================================================
# 300 DPI is the minimum for journal submission. We set it globall so every
# figure saved in this notebook meets publication standards automatically.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

# Colour palette - consistent across all figures so read can tract
# strategies and models visually across the paper without re-reading legends.
# Colours are from the matplotlib default cycle.
PALETTE: dict = {
    "Pooled baseline":      "#1f77b4",   # Blue
    "Age-inclusive pooled": "#2ca02c",   # Green
    "Age-stratified":       "#d62728",   # Red
    "Random Forest":        "#1f77b4",   # Blue
    "ANN":                  "#ff7f0e",   # Orange
    "No caries":            "#7f7f7f",   # Grey
    "Mild caries":          "#ffbf00",   # Amber
    "Severe caries":        "#d62728",   # Red
    "Caries present":       "#d62728",   # Red
    }

#===============================================================================
# FINAL CONFIGURATION SUMMARY
#===============================================================================
print("\n" + "=" * 60)
print("DASC512 ASSIGNMENT 2 — CONFIGURATION SUMMARY")
print("=" * 60)
print(f"  Seed              : {SEED}")
print(f"  CV folds          : {N_SPLITS}")
print(f"  Quick mode        : {QUICK_MODE}")
print(f"  Run multiclass    : {RUN_MULTICLASS}")
print(f"  RF estimators     : {RF_N_ESTIMATORS}")
print(f"  ANN epochs        : {ANN_EPOCHS}")
print(f"  Imputer max iter  : {IMPUTER_MAX_ITER}")
print(f"  ANN hidden layers : {ANN_HIDDEN_LAYERS}")
print(f"  ANN dropout       : {ANN_DROPOUT}")
print(f"  ANN weight decay  : {ANN_WEIGHT_DECAY}")
print("=" * 60)
print("Run Cell 0 (linting setup) before this cell.")
if QUICK_MODE:
    print("QUICK_MODE = True. Smoke test only.")
else:
    print("QUICK_MODE = False. Full submission run.")

Mounted at /content/drive
Google Drive mounted successfully.
Project directory : /content/drive/MyDrive/DASC512_Assessment2
Dataset expected  : /content/drive/MyDrive/DASC512_Assessment2/all_2018_processed_cleaned_data.txt
Figures will save : /content/drive/MyDrive/DASC512_Assessment2/outputs/figures

DASC512 ASSIGNMENT 2 — CONFIGURATION SUMMARY
  Seed              : 42
  CV folds          : 5
  Quick mode        : False
  Run multiclass    : True
  RF estimators     : 200
  ANN epochs        : 100
  Imputer max iter  : 5
  ANN hidden layers : (128, 64)
  ANN dropout       : 0.3
  ANN weight decay  : 0.0001
Run Cell 0 (linting setup) before this cell.
QUICK_MODE = False. Full submission run.


In [3]:
#===============================================================================
# CHAPTER 2: DATA LOADING AND VALIDATION
#===============================================================================
# WHY THIS DATASET?
# This is the cleaned output of the STAR pipeline from Orlenko et al.
# (2025). Orlenko applied outlier removal (STAR algorithm), domain-relevant
# variable selection, one-hot encoding of categorical variables, and
# documented the result as 438 variables across 8,099 participants from
# NHANES 2017-2018. We use this cleaned version directly - replicating
# their preprocessing - so our predictive models are a direct extension
# of their unsupervised phenotyping work rather than a parallel study.
#
#===============================================================================
# FILE EXISTS CHECK
#===============================================================================
# A clear error message here saves significant debugging time.
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at:\n  {DATA_PATH}\n\n"
        f"Upload all_2018_processed_cleaned_data.txt to:\n"
        f"  Google Drive > DASC512_Assessment2 > "
        f"all_2018_processed_cleaned_data.txt\n\n"
        f"Dataset is the cleaned NHANES 2017-2018 output from:\n"
        f"  Orlenko et al. (2025) doi:10.1177/00220345251398027"
    )

#===============================================================================
# LOAD
#===============================================================================
print("Loading cleaned NHANES 2017-2018 dataset...")
raw_df = pd.read_csv(DATA_PATH, sep="\t", low_memory=False)
print(f"Raw dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")

#===============================================================================
# STRUCTURAL ASSERTIONS
#===============================================================================
# These confirm we have the right file before spending any time on it.
# If any assertion fails, the message tells us exactly what went wrong
assert raw_df.shape[0] > 5000, (
    f"Expected >5000 rows, got {raw_df.shape[0]:,}. "
    f"Dataset may be truncated or wrong file."
)
assert raw_df.shape[1] > 400, (
    f"Expected >400 columns after Orlenko cleaning, "
    f"got {raw_df.shape[1]}."
)

# outcome_s: the raw caries score we derive all outcomes from.
# RIDAGEYR: age in years, used to assign Orlenko age strata.
required_columns = ["outcome_s", "outcome_b", "RIDAGEYR"]
missing_cols = [c for c in required_columns if c not in raw_df.columns]
assert not missing_cols, (
    f"Required columns missing from dataset: {missing_cols}\n"
    f"Check that you uploaded the correct file."
)

#===============================================================================
# OUTCOME INSPECTION
#===============================================================================
# We inspect the raw outcome distributions, so that we know exactly what we are
# working with before delivering new columns.
print("\noutcome_s (raw caries score, 0-10) summary:")
print(raw_df["outcome_s"].describe().round(3).to_string())
print(f"\nRows with missing outcome_s : "
      f"{raw_df['outcome_s'].isna().sum():,} of {len(raw_df):,}")
print(f"Rows with missing outcome_b : "
      f"{raw_df['outcome_b'].isna().sum():,} of {len(raw_df):,}")

#===============================================================================
# AGE INSPECTION
#===============================================================================
print(f"\nRIDAGEYR (age in years) summary:")
print(raw_df["RIDAGEYR"].describe().round(1).to_string())
print(f"Rows with missing RIDAGEYR  : "
      f"{raw_df['RIDAGEYR'].isna().sum():,} of {len(raw_df):,}")

#===============================================================================
# OVERALL MISSINGNESS
#===============================================================================
# The Orlenko pipeline removed variables with >50% missingness.
# Nevertheless, overall missingesness rate of what remains is still being
# reported - this number motivates the use of iterative imputation in
# Chapter 5.
total_cells = raw_df.size
missing_cells = int(raw_df.isna().sum().sum())
print(f"\nOverall missingness : "
      f"{missing_cells:,} / {total_cells:,} cells "
      f"({missing_cells / total_cells * 100:.1f}%)")
print(f"Columns with any missing value : "
      f"{int((raw_df.isna().sum() > 0).sum())} of {raw_df.shape[1]}")

#===============================================================================
# LEAKAGE PRE-CHECK
#===============================================================================
# outcome_b is a pre-computed binary caries indicator that perfectly
# matches our derived caries_binary column. It must NEVER appear as a
# predictor.
# This check documents awareness - the exclusion happens in Chapter 8.
assert "outcome_b" in raw_df.columns, (
    "outcome_b column not found. Check dataset version."
)
print(f"\noutcome_b present and will be excluded from features: confirmed.")
print(f"outcome_b / outcome_s agreement check will run in Chapter 3.")

print("\nData loaded and validated. Proceed to Chapter 3.")

Loading cleaned NHANES 2017-2018 dataset...
Raw dataset shape: 9,254 rows x 440 columns

outcome_s (raw caries score, 0-10) summary:
count    8186.000
mean        1.204
std         2.141
min         0.000
25%         0.000
50%         0.000
75%         2.000
max        10.000

Rows with missing outcome_s : 1,068 of 9,254
Rows with missing outcome_b : 888 of 9,254

RIDAGEYR (age in years) summary:
count    9254.0
mean       34.3
std        25.5
min         0.0
25%        11.0
50%        31.0
75%        58.0
max        80.0
Rows with missing RIDAGEYR  : 0 of 9,254

Overall missingness : 1,015,921 / 4,071,760 cells (25.0%)
Columns with any missing value : 378 of 440

outcome_b present and will be excluded from features: confirmed.
outcome_b / outcome_s agreement check will run in Chapter 3.

Data loaded and validated. Proceed to Chapter 3.


In [4]:
#===============================================================================
# CHAPTER 3: OUTCOME ENGINEERING AND AGE STRATIFICATION
#===============================================================================
# PURPOSE OF THIS CHAPTER
# Build the two prediction targets and assign every participant to one of
# Orlenko's four age strata. Then prove that five-fold stratified
# cross-validation is mathematically feasible for every combination before
# any model is trained. A feasibility failure here is caught in seconds;
# the same failure discovered mid-experiment costs hours
#
# WHY WE DROP ON outcome_s RATHER THAN outcome_b:
# Both outcomes (binary and multiclass severity) are derived from outcome_s.
# Dropping on outcome_s ensures both targets are defined for every row we
# analyse - a row with outcome_s cannot have a missing severity class.
# The 180 additional rows available via outcome_b alone lack a severity
# score and cannot contribute to the multiclass analysis. We therefore
# exclude them from both analyses for consistency and document this as a
# minor limitation: all 180 have outcome_b = 1 (caries present), so their
# exclusion produces a marginal downard bias in estimated prevalence.
#===============================================================================

print(f"Rows before dropping missing outcome_s : {len(raw_df):,}")
df = raw_df.dropna(subset=["outcome_s"]).copy()
removed = len(raw_df) - len(df)
print(f"Rows after  dropping missing outcome_s : {len(df):,} "
      f"(removed {removed:,}, {removed / len(raw_df) * 100:.1f}%)")

#===============================================================================
# OUTCOME 1 - Binary caries presence (PRIMARY)
#===============================================================================
# Any outcome_s >=1 means at least one affected surface or tooth.
# This binary threshold matches every major published caries prediction
# study that uses NHANES (Pang et al. 2021; Tirkkonen et al. 2024),
# which allows direct AUC comaprison with the existing literature.
# It is also the most robust clinical threshold: presence vs absence is
# the most consistently applied judgement in NHANES, minimising
# the inter-examiner variability as a noise source
df["caries_binary"] = (df["outcome_s"] >= 1).astype(int)

#===============================================================================
# LEAKAGE VERIFICATION - outcome_b matches caries_binary exactly
#===============================================================================
# We established in Chapter 2 that outcome_b must be excluded from features.
# Here we confirm WHY: it is a perfect copy of caries_binary (correlation
# = 1.0, zero mismatches). Any model that sees outcome_b as a feature
# achieves AUC = 1.0 trivially - not by learning clinical patterns but by
# reading the answer off a copy of the label. This is data leakage.
# The exclusion is enforced in Chapter 8 via _NEVER_FEATURES.
rows_with_both = df["outcome_b"].notna().sum()
mismatches = (
    df.loc[df["outcome_b"].notna(), "outcome_b"].astype(int)
    != df.loc[df["outcome_b"].notna(), "caries_binary"]
).sum()
print(f"\nLeakage verification:")
print(f"  Rows where both outcome_b and caries_binary are defined : "
      f"{rows_with_both:,}")
print(f"  Mismatches between outcome_b and caries_binary          : "
      f"{mismatches}")
print(f"  outcome_b is a perfect copy of caries_binary            : "
      f"{mismatches == 0}")
print(f"  Conclusion: outcome_b excluded from all feature sets (Chapter 8)")

#===============================================================================
# OOUTCOME 2 - Multiclass severity (SECONDARY)
#===============================================================================
# Three clinical classes preserve more information than binary alone
# 0 = No caries (outcome_s == 0)
# 1 = Mild caries (1 <= outcome_s <=3)
# 2 = Severe caries (outcome_s >=4)
#
# The severe threshold (>=4 affected sufaces/teeth) follows the
# established Severe Early Childhood Caries (S-ECC) definition
# (Tinanoff et al. 2019), adopted by Orlenko as the severity boundary.
# This secondary outcome lets us ask whether models can distibguish
# severity, not just presence - a clinically meaningful question for
# treatment planning and resource allocation.

def categorise_severity(score: float) -> int:
    """Map a raw caries score to a three-class clinical severity label.

    Uses the S-ECC threshold (Tinanoff et al. 2019) for the severe
    boundary, consistent with Orlenko et al. (2025).

    Args:
        score: Raw outcome_s value (0 to 10).

    Returns:
        0 for no caries, 1 for mild (1-3), 2 for severe (>=4).
    """
    if score == 0:
        return 0
    if score <= 3:
        return 1
    return 2


df["caries_severity"] = df["outcome_s"].apply(categorise_severity)

#===============================================================================
# AGE STRATA = exactly as Orlenko et al. (2025) defined them
#===============================================================================
# Orlenko's central finding was that children (<=5y) and seniors (>=65y)
# exhibit the largest, most distinct variable clusters - suggesting
# different aetiological pathwats. We use identical boundaries so our
# predictive models direct test whether those phenotypic differences
# translate into prediction perforamnce differences.

def assign_age_stratum(age: float) -> str:
    """Assign a participant to an Orlenko age stratum.

    Boundaries are taken verbatim from Orlenko et al. (2025) Table 1
    to ensure direct comparability with their unsupervised findings.

    Args:
        age: Age in years from RIDAGEYR.

    Returns:
        One of 'Children', 'Youth', 'Adults', 'Seniors'.
    """
    if age <= 5:
        return "Children"
    if age <= 18:
        return "Youth"
    if age <= 64:
        return "Adults"
    return "Seniors"


df["age_stratum"] = df["RIDAGEYR"].apply(assign_age_stratum)
AGE_STRATA = ["Children", "Youth", "Adults", "Seniors"]

#===============================================================================
# DISTRIBUTION REPORT
#===============================================================================
print("\nBinary outcome distribution:")
for val, count in df["caries_binary"].value_counts().sort_index().items():
    label = "No caries" if val == 0 else "Caries present"
    print(f"  {val} ({label:<14}): {count:,} ({count/len(df)*100:.1f}%)")

print("\nSeverity outcome distribution:")
severity_labels = {0: "No caries", 1: "Mild    ", 2: "Severe  "}
for val, count in df["caries_severity"].value_counts().sort_index().items():
    print(f"  {val} ({severity_labels[val]}): "
          f"{count:,} ({count/len(df)*100:.1f}%)")

print("\nAge stratum sizes and caries prevalence:")
for stratum in AGE_STRATA:
    sub = df[df["age_stratum"] == stratum]
    n = len(sub)
    prev = sub["caries_binary"].mean() * 100
    severe = (sub["caries_severity"] == 2).mean() * 100
    print(f"  {stratum:<9}: n={n:,}  "
          f"binary prevalence={prev:5.1f}%  "
          f"severe prevalence={severe:4.1f}%")

#===============================================================================
# CROSS-VALIDATION FEASIBILITY CHECK
#===============================================================================
# Stratified K-fold requires at least K samples of every class in every
# stratum. With K=5, any class with fewer than 5 cases anywhere would
# silently break the analysis. We verify this now - before any model is
# trained - so a failure costs seconds not hours

def assert_cv_feasible(
    frame: pd.DataFrame,
    outcome_col: str,
    group_col: str = None,
    k: int = N_SPLITS,
) -> None:
    """Raise AssertionError if stratified k-fold is not feasible.

    Args:
        frame: DataFrame to check.
        outcome_col: Name of the outcome column.
        group_col: If provided, check feasibility within each group.
        k: Number of folds. Default is N_SPLITS from Chapter 1.
    """
    if group_col is None:
        smallest = frame[outcome_col].value_counts().min()
        assert smallest >= k, (
            f"'{outcome_col}' has a class with only {smallest} cases "
            f"(need >= {k} for {k}-fold CV)."
        )
    else:
        for name, grp in frame.groupby(group_col):
            smallest = grp[outcome_col].value_counts().min()
            assert smallest >= k, (
                f"In stratum '{name}', '{outcome_col}' has a class with "
                f"only {smallest} cases (need >= {k})."
            )


assert_cv_feasible(df, "caries_binary")
assert_cv_feasible(df, "caries_severity")
assert_cv_feasible(df, "caries_binary", group_col="age_stratum")
assert_cv_feasible(df, "caries_severity", group_col="age_stratum")

print("\nFeasibility check passed for all outcomes and all age strata.")
print("Proceed to Chapter 4.")

Rows before dropping missing outcome_s : 9,254
Rows after  dropping missing outcome_s : 8,186 (removed 1,068, 11.5%)

Leakage verification:
  Rows where both outcome_b and caries_binary are defined : 8,186
  Mismatches between outcome_b and caries_binary          : 0
  outcome_b is a perfect copy of caries_binary            : True
  Conclusion: outcome_b excluded from all feature sets (Chapter 8)

Binary outcome distribution:
  0 (No caries     ): 5,099 (62.3%)
  1 (Caries present): 3,087 (37.7%)

Severity outcome distribution:
  0 (No caries): 5,099 (62.3%)
  1 (Mild    ): 2,017 (24.6%)
  2 (Severe  ): 1,070 (13.1%)

Age stratum sizes and caries prevalence:
  Children : n=921  binary prevalence= 14.1%  severe prevalence= 6.5%
  Youth    : n=2,034  binary prevalence= 22.4%  severe prevalence= 5.2%
  Adults   : n=3,942  binary prevalence= 45.3%  severe prevalence=13.8%
  Seniors  : n=1,289  binary prevalence= 55.4%  severe prevalence=28.0%

Feasibility check passed for all outcomes and 

In [5]:
#===============================================================================
# CHAPTER 4: EXPLORATORY DATA ANALYSIS
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Produce Figure 1: the visual that motivates everything that follows.
# This figure shows why a single pooled model is insufficient and why age
# stratification is necessary.
#
# WHAT FIGURE 1 SHOWS:
# Panel A: Binary caries prevalence by age stratum - the monotonic gradient
#          from 14% (children) to 55% (seniors) that Orlenko's clustering
#          identified structurally
# Panel B: Severity distribution by age stratum - shows that the disease is
#          not just more prevalent in seniors but qualitatively more severe,
#          which a pooled model cannot represent adequately
#
# WHY WITHIN-STRATUM PERCENTAGES, NOT COUNTS?
# Strata have very different sizes (Children n=921 vs Adults n=3,942).
# Raw counts would make Adults dominate visually. Within-stratum
# percentages make all four groups visually comparable, which is the
# honest representation.
#===============================================================================

def save_figure(figure: plt.Figure, filename: str) -> str:
    """Save a figure to the figures directory at 300 DPI.

    Args:
        figure: The matplotlib Figure object to save.
        filename: File name including extension (e.g. 'Figure_1.png').

    Returns:
        Full path of the saved file.
    """
    path = os.path.join(FIGURE_DIR, filename)
    figure.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(figure)
    print(f"  Saved: {filename}")
    return path


def within_group_percentages(
    frame: pd.DataFrame,
    group_col: str,
    value_col: str,
) -> pd.DataFrame:
    """Compute within-group percentages for a categorical column.

    Each group sums to 100%. This normalisation makes groups with
    different sample sizes visually comparable in bar charts.

    Args:
        frame: Source DataFrame.
        group_col: Column defining the groups (e.g. 'age_stratum').
        value_col: Categorical column to compute percentages for.

    Returns:
        Long-format DataFrame with columns [group_col, value_col,
        'count', 'percentage'].
    """
    counts = (
        frame.groupby([group_col, value_col])
        .size()
        .reset_index(name="count")
    )
    group_totals = counts.groupby(group_col)["count"].transform("sum")
    counts["percentage"] = counts["count"] / group_totals * 100.0
    return counts

#===============================================================================
# BUILD FIGURE 1
#===============================================================================

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

# Panel A — Binary prevalence
bin_counts = within_group_percentages(df, "age_stratum", "caries_binary")
bin_counts["Status"] = bin_counts["caries_binary"].map(
    {0: "No caries", 1: "Caries present"}
)
sns.barplot(
    data=bin_counts,
    x="age_stratum",
    y="percentage",
    hue="Status",
    order=AGE_STRATA,
    hue_order=["No caries", "Caries present"],
    palette=[PALETTE["No caries"], PALETTE["Caries present"]],
    ax=axes[0],
)
axes[0].set_title("A  Binary caries prevalence by age stratum",
                  fontweight="bold")
axes[0].set_xlabel("Age stratum (Orlenko et al. 2025 boundaries)")
axes[0].set_ylabel("Percentage within stratum (%)")
axes[0].set_ylim(0, 100)
axes[0].legend(title="Status", loc="upper left")

# Annotate the prevalence values on each caries-present bar so the
# gradient is immediately readable without consulting the axis.
for patch, stratum in zip(axes[0].patches[len(AGE_STRATA):], AGE_STRATA):
    sub = df[df["age_stratum"] == stratum]
    prev = sub["caries_binary"].mean() * 100
    axes[0].text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + 1.5,
        f"{prev:.1f}%",
        ha="center", va="bottom", fontsize=8, fontweight="bold",
    )

# Panel B — Severity distribution
sev_counts = within_group_percentages(df, "age_stratum", "caries_severity")
sev_counts["Severity"] = sev_counts["caries_severity"].map(
    {0: "No caries", 1: "Mild caries", 2: "Severe caries"}
)
sns.barplot(
    data=sev_counts,
    x="age_stratum",
    y="percentage",
    hue="Severity",
    order=AGE_STRATA,
    hue_order=["No caries", "Mild caries", "Severe caries"],
    palette=[
        PALETTE["No caries"],
        PALETTE["Mild caries"],
        PALETTE["Severe caries"],
    ],
    ax=axes[1],
)
axes[1].set_title("B  Caries severity distribution by age stratum",
                  fontweight="bold")
axes[1].set_xlabel("Age stratum (Orlenko et al. 2025 boundaries)")
axes[1].set_ylabel("Percentage within stratum (%)")
axes[1].set_ylim(0, 100)
axes[1].legend(title="Severity", loc="upper left")

#===============================================================================
# FIGURE CAPTION
#===============================================================================
caption = (
    "Dental caries prevalence across Orlenko age strata, "
    "NHANES 2017-2018 (n=8,186 after exclusions). "
    "Panel A: binary prevalence rises monotonically from 14.1% "
    "(Children) to 55.4% (Seniors), motivating age-stratified modelling. "
    "Panel B: severe caries (≥4 affected surfaces, Tinanoff et al. 2019) "
    "is disproportionately concentrated in Seniors (28.0%), suggesting "
    "distinct aetiological pathways across the life course "
    "(Orlenko et al. 2025)."
)
fig.text(
    0.5, -0.06, caption,
    ha="center", va="top", fontsize=8,
    wrap=True, style="italic",
    transform=fig.transFigure,
)

fig.tight_layout()
FIG1_PATH = save_figure(fig, "Figure_1_Caries_Prevalence_By_Age.png")

#===============================================================================
# SUMMARY STATISTICS FOR REPORT USE
#===============================================================================
print("\nKey numbers for your Methods/Results sections:")
print(f"  Total analysed        : {len(df):,}")
print(f"  Overall binary prev.  : {df['caries_binary'].mean()*100:.1f}%")
print(f"  Overall severe prev.  : "
      f"{(df['caries_severity']==2).mean()*100:.1f}%")
print(f"  Excluded (no outcome) : 1,068 (11.5% of raw 9,254)")
print(f"  Of excluded, caries+  : 180 (all outcome_b=1) — see Methods")
print("\nChapter 4 complete. Figure 1 saved.")
print("Proceed to Chapter 5.")

  Saved: Figure_1_Caries_Prevalence_By_Age.png

Key numbers for your Methods/Results sections:
  Total analysed        : 8,186
  Overall binary prev.  : 37.7%
  Overall severe prev.  : 13.1%
  Excluded (no outcome) : 1,068 (11.5% of raw 9,254)
  Of excluded, caries+  : 180 (all outcome_b=1) — see Methods

Chapter 4 complete. Figure 1 saved.
Proceed to Chapter 5.


In [6]:
#===============================================================================
# CHAPTER 5: PREPROCESSING UTILITIES
#===============================================================================
# PURPOSE OF THIS CHAPTER
# Define the preprocessing pipeline that runs inside every cross-validation
# fold.
#
# THE CENTRAL PROBLEM - DATA LEAKAGE:
# Data leakage occurs when information from the validation fold influences how
# the training fold is processed. The most common form is fitting an imputer or
# scaler on the entire dataset before splitting - the imputed values in the
# training fold then reflect statistical patterns from the validation patients,
# making the model appear better than it is on genuinely unseen data.
#
# This is not a theoretical concern. Tirkonnen et al. (2024) documented an AUC
# of 0.785 internally collapsing to 0.550 on external validation - a likely
# consequence of preprocessing leakage. This is prevent by fitting the
# imputer and scaler exclusively on the training fold and applying the fitted
# objects to the validation fold.
#
# IMPUTE-ONCE DESIGN:
# Both models (Random Forest and ANN) are served from a single imputation
# per fold. This halves the most expensive operation and - more importantly -
# guarantees that both models see identically imputed data on identical
# splits. This makes the paired Nadeau-Bengio statistical comparison in
# Chapter 10 strictly valid: any performance difference between models cannot
# be attributed to different imputed values.
#
# WHY ITERATIVE IMPUTATION (MICE-STYLE)?
# With 25% overall missingness across 378 columns, simple strategies
# (mean, median, mode imputation) ignore the multivariate structure of the
# data. A patient's missing iron level is predictable from their
# hb, age, and dietary intake. Iterative imputation
# (van Buuren & Groothuis-Oudshoorn 2011) models each missing variable
# as a function of all others in a repeated cycle, producing imputations
# that respect the correlation structure of NHANES - the same approach
# used in the Orlenko pipeline.
#===============================================================================

def preprocess_fold(
    X_train_df: pd.DataFrame,
    X_valid_df: pd.DataFrame,
    seed: int,
) -> tuple:
    """Impute once per fold, return arrays ready for both RF and ANN.

    The iterative imputer is fitted on the training fold only, then
    applied to the validation fold. A StandardScaler is also fitted on
    the training fold only. The validation fold never influences either
    fitted object — this is what makes the pipeline leak-proof.

    Both models receive identically imputed data (same fold, same
    imputer fit). The ANN additionally receives standardised data
    because neural networks are sensitive to feature scale — gradient
    updates are dominated by large-magnitude features if inputs are
    not normalised. Random Forest is scale-invariant by construction
    (splits are based on rank order, not absolute values), so it
    receives the imputed-only version.

    Args:
        X_train_df: Training-fold feature DataFrame (raw, with NaNs).
        X_valid_df: Validation-fold feature DataFrame (raw, with NaNs).
        seed: Random seed for the iterative imputer's internal models.

    Returns:
        Tuple of four float32 arrays:
            X_tr_rf  — imputed only, for Random Forest training.
            X_va_rf  — imputed only, for Random Forest validation.
            X_tr_ann — imputed + standardised, for ANN training.
            X_va_ann — imputed + standardised, for ANN validation.
    """
    # Work on copies. Replace infinities with NaN so the imputer treats
    # them as missing rather than as extreme valid values.
    X_tr = X_train_df.copy().replace([np.inf, -np.inf], np.nan)
    X_va = X_valid_df.copy().replace([np.inf, -np.inf], np.nan)

    # Edge case: in small strata a column can be entirely missing within
    # the training fold (e.g. a lab test recorded only for adults, inside
    # the Children stratum). The imputer cannot learn from a column it
    # never observes. We set such columns to zero using ONLY training-fold
    # information, then apply the same to validation — no leakage.
    empty_cols = X_tr.columns[X_tr.isna().all()].tolist()
    if empty_cols:
        print(f"    Warning: {len(empty_cols)} columns entirely missing "
              f"in training fold — set to 0.")
        X_tr.loc[:, empty_cols] = 0.0
        X_va.loc[:, empty_cols] = 0.0

    # ---- Iterative (MICE-style) imputation — fitted on training only ----
    # n_nearest_features: limits each imputation model to the most
    # correlated predictors. With 438 candidates, using all of them makes
    # each imputation model expensive and adds little accuracy. Limiting
    # to 20-30 neighbours preserves the statistical behaviour of chained
    # imputation while keeping runtime feasible on Colab's free tier.
    # initial_strategy='median': robust starting point for NHANES
    # distributions which are heavily right-skewed (e.g. lead, cotinine).
    # skip_complete=True: skips columns with no missing values — no
    # computation needed where there is nothing to impute.
    n_feat = X_tr.shape[1]
    nearest = min(IMPUTER_NEAREST_FEATURES, max(n_feat - 1, 1))
    imputer = IterativeImputer(
        max_iter=IMPUTER_MAX_ITER,
        random_state=seed,
        initial_strategy="median",
        n_nearest_features=nearest,
        skip_complete=True,
        sample_posterior=False,
    )
    # FIT on training fold only
    X_tr_imp = imputer.fit_transform(X_tr)
    # APPLY the fitted imputer to validation — no refitting
    X_va_imp = imputer.transform(X_va)

    # Random Forest receives imputed-only arrays.
    # cast to float32 to reduce memory footprint.
    X_tr_rf = X_tr_imp.astype(np.float32)
    X_va_rf = X_va_imp.astype(np.float32)

    # ---- Standardisation for the ANN — fitted on training only ----
    # StandardScaler subtracts the training mean and divides by training
    # standard deviation. Fitting on training only means the validation
    # fold is scaled using training statistics — not its own statistics.
    # Zero-variance columns (std=0) are handled internally by sklearn:
    # the scale is set to 1 so those columns pass through unchanged.
    scaler = StandardScaler()
    # FIT on training fold only
    X_tr_ann = scaler.fit_transform(X_tr_imp).astype(np.float32)
    # APPLY the fitted scaler to validation
    X_va_ann = scaler.transform(X_va_imp).astype(np.float32)

    # Post-imputation NaN check — catches edge cases where the imputer
    # itself produces NaN (extremely rare but would propagate silently
    # through the ANN as NaN loss values).
    if np.isnan(X_tr_ann).any() or np.isnan(X_va_ann).any():
        raise ValueError(
            "NaN values detected after imputation and scaling. "
            "Check for columns with extreme missingness in this fold."
        )

    return X_tr_rf, X_va_rf, X_tr_ann, X_va_ann


print("Leak-proof preprocessing function defined.")
print("Key design properties:")
print("  Imputer  : fitted on training fold only")
print("  Scaler   : fitted on training fold only")
print("  Both models receive identically imputed data per fold")
print("  NaN check runs after every imputation")
print("Proceed to Chapter 6.")

Leak-proof preprocessing function defined.
Key design properties:
  Imputer  : fitted on training fold only
  Scaler   : fitted on training fold only
  Both models receive identically imputed data per fold
  NaN check runs after every imputation
Proceed to Chapter 6.


In [7]:
#===============================================================================
# CHAPTER 6: MODEL ARCHITECTURE DEFINITIONS
#===============================================================================
# PURPOSE OF THIS CHAPTER
# Define the Random Forest (shallow learner) and the Artificial Neural
# Network (deep learner). Every architectural decision is justified here.
#
# MODEL 1 - RANDOM FOREST (Shallow Learning)
#------------------------------------------
# A Random Forest is an ensemble of decision trees. Understanding it requires
# understanding three mechanisms working together:
#
# 1. DECISION TREE (the base unit):
#    A decision tree splits patients into groups by asking binary questions
#    about features: "Is blood lead > 2.1 µg/dL? Yes -> left branch.
#    No -> right branch." At each split, the algorithm searches all features
#    and all possible thresholds to find the split that best separates
#    caries-positive from caries-negative patients. "Best" is measured by
#    Gini impurity - how mixed the two resulting groups are.
#    A pure group (all caries or all no-caries) has Gini = 0
#
# 2. BOOTSTRAP AGGREGATING (Bagging):
#    A single deep decision tree memorises training data (overfitting).
#    The forest prevents this by training each tree on a different
#    bootstrap sample - a random draw with replacement from the training
#    set, typically ~63% unoque patients. Trees trained on different samples
#    make different errors. Averaging their predictions cancels individual
#    errors - the forest is more accurate than any single tree.
#
# 3. RANDOM FOREST SELECTION:
#    At each split, only a random subset of features is conasidered
#    (default: sqrt(438) ~ 21 features). This decorrelates the trees -
#    if blood lead is the strongest predictor, every tree would split on it
#    first without this constraint, making all trees similar and defeating
#    the purpose of aggregation
#
# WHY RANDOM FOREST FOR THIS DATASET?
# - Handles mixed data type (continuous lab values, binary one-hot
# encorded dietary variables) without transformation beyond imputation.
# - Scale-invariant: does not require standardisation
# - Naturally produces feature importance (mean decrease in Gini impurity),
# which directly connects to Orlenko's variable clusters.
# - Already benchmarked in caries prediction: Pang et al. (2021)
#   achieved AUC 0.73-0.78 on NHANES with Random Forest, giving us a
#   direct external comparison point.
# - n_estimators=200 (full run): 200 trees is standard for tabular
#   health data — performance plateaus beyond this for NHANES-scale data.
#
# MODEL 2 — ARTIFICIAL NEURAL NETWORK (Deep Learning)
# ----------------------------------------------------
# A feed-forward ANN passes data through successive layers of
# mathematical transformations to learn non-linear relationships that
# a single decision boundary cannot capture.
#
# ARCHITECTURE: Input(438) → [Linear→BatchNorm→ReLU→Dropout] x2 → Output
#
# Each layer component has a specific role:
#
# LINEAR LAYER (nn.Linear):
#   Computes output = W·input + b, where W is a weight matrix and b is
#   a bias vector. The weights are the parameters the network learns.
#   With input dimension 438 and first hidden layer 128, this is a
#   438×128 weight matrix — 56,064 learnable parameters in one layer.
#
# BATCH NORMALISATION (nn.BatchNorm1d):
#   Normalises each feature across the batch to zero mean and unit
#   variance before activation. Without this, deep layers receive inputs
#   that shift dramatically as weights update — making training unstable.
#   BatchNorm stabilises training and allows higher learning rates.
#   IMPORTANT: requires batch size > 1. We use drop_last=True in the
#   DataLoader to prevent a final batch of size 1 crashing training.
#
# RELU ACTIVATION (nn.ReLU):
#   Applies f(x) = max(0, x). Without a non-linear activation function,
#   stacking linear layers produces only another linear transformation —
#   the network cannot learn complex patterns regardless of depth. ReLU
#   is the standard choice: computationally cheap, avoids the vanishing
#   gradient problem that plagued earlier activations (sigmoid, tanh).
#
# DROPOUT (nn.Dropout, p=0.30):
#   Randomly sets 30% of activations to zero during each training pass.
#   This forces the network to learn redundant representations — no
#   single neuron can carry critical information alone. Acts as a strong
#   regulariser. With 65,090 parameters and only ~736 Children training
#   samples per fold, dropout is essential to prevent memorisation.
#   Dropout is disabled automatically during model.eval() — inference
#   uses all neurons.
#
# OUTPUT LAYER:
#   Final nn.Linear produces raw class scores (logits). Logits are
#   passed directly to nn.CrossEntropyLoss, which applies softmax
#   internally. Applying softmax manually before the loss function is
#   a common silent bug — it produces numerically unstable results.
#   Probabilities for evaluation are computed separately via
#   torch.softmax() after training.
#===============================================================================

#===============================================================================
# RANDOM FOREST FACTORY
#===============================================================================
def make_random_forest(seed: int = SEED) -> "RandomForestClassifier":
    """Create a configured but untrained Random Forest.

    All hyperparameters are set here. The only argument is the seed,
    which ensures different folds use different (but reproducible)
    random states for bootstrap sampling and feature selection.

    Args:
        seed: Reproducibility seed for bootstrap and feature sampling.

    Returns:
        An untrained RandomForestClassifier ready for .fit().
    """
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        # 'balanced': inversely weights each class by its frequency.
        # Formula: weight_c = n_samples / (n_classes * n_samples_c).
        # With Children stratum ~14% positive, this upweights caries
        # errors by ~6x — the model is penalised heavily for missing
        # a caries-positive child.
        class_weight=RF_CLASS_WEIGHT,
        random_state=seed,
        # n_jobs=-1: use all available CPU cores. Parallelises tree
        # building — each tree is independent so parallelisation is
        # trivial and safe.
        n_jobs=-1,
    )


#===============================================================================
# NEURAL NETWORK ARCHITECTURE
#===============================================================================
class CariesANN(nn.Module):
    """Feed-forward ANN for dental caries classification.

    Architecture: Linear → BatchNorm → ReLU → Dropout (repeated for
    each hidden layer) → final Linear output layer.

    Inherits from nn.Module, which is PyTorch's base class for all
    neural networks. Two methods are required:
      __init__: defines the layers.
      forward:  defines how data flows through them.
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_layers: tuple = ANN_HIDDEN_LAYERS,
        dropout: float = ANN_DROPOUT,
    ) -> None:
        """Build the network layer stack.

        Args:
            input_dim: Number of input features (438 after exclusions).
            output_dim: Number of output classes (2 binary, 3 severity).
            hidden_layers: Tuple of hidden layer widths, e.g. (128, 64).
            dropout: Fraction of units dropped per forward pass (0.30).
        """
        super().__init__()
        blocks = []
        prev_dim = input_dim
        for width in hidden_layers:
            blocks.extend([
                nn.Linear(prev_dim, width),
                nn.BatchNorm1d(width),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = width
        # Final layer: no activation — raw logits for CrossEntropyLoss
        blocks.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: data flows through the layer stack.

        Args:
            x: Input tensor of shape (batch_size, input_dim).

        Returns:
            Raw logits of shape (batch_size, output_dim).
            Do NOT apply softmax here — CrossEntropyLoss does it
            internally. Applying it twice is a silent numerical bug.
        """
        return self.network(x)


#===============================================================================
# CLASS WEIGHT UTILITY FOR THE ANN LOSS FUNCTION
#===============================================================================
def torch_class_weights(
    y_train: np.ndarray,
    n_classes: int,
) -> torch.Tensor:
    """Compute balanced class weights for nn.CrossEntropyLoss.

    Mirrors sklearn's 'balanced' formula exactly:
        weight_c = n_samples / (n_classes * n_samples_c)
    so RF and ANN apply identical class weighting logic and their
    performance differences reflect architecture, not weighting.

    Args:
        y_train: Integer training labels for this fold.
        n_classes: Total number of classes.

    Returns:
        FloatTensor of shape (n_classes,) for CrossEntropyLoss(weight=).
    """
    counts = np.bincount(y_train.astype(int), minlength=n_classes)
    n_total = counts.sum()
    weights = np.zeros(n_classes, dtype=np.float32)
    for c, n_c in enumerate(counts):
        if n_c > 0:
            weights[c] = n_total / (n_classes * n_c)
        # If a class has 0 training samples, weight stays 0 —
        # the loss ignores classes not seen in this fold.
    return torch.tensor(weights, dtype=torch.float32)


#===============================================================================
# ANN TRAINING LOOP
#===============================================================================
def train_ann(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    y_va: np.ndarray,
    n_classes: int,
    seed: int,
) -> tuple:
    """Train the ANN for one fold, return the model and loss history.

    Training follows the Week 6 practical structure: DataLoader with
    batch_size=32, Adam optimiser, CrossEntropyLoss, fixed epochs,
    per-epoch training and validation loss recorded.

    Args:
        X_tr: Scaled training features, shape (n_train, n_features).
        y_tr: Integer training labels, shape (n_train,).
        X_va: Scaled validation features, shape (n_val, n_features).
        y_va: Integer validation labels, shape (n_val,).
        n_classes: Number of output classes (2 or 3).
        seed: Reproducibility seed for weight init and batch shuffling.

    Returns:
        Tuple of (trained CariesANN, loss history DataFrame).
        History has columns: epoch, train_loss, validation_loss, fold.
    """
    set_all_seeds(seed)

    # Convert numpy arrays to PyTorch tensors
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.astype(int), dtype=torch.long)
    X_va_t = torch.tensor(X_va, dtype=torch.float32)
    y_va_t = torch.tensor(y_va.astype(int), dtype=torch.long)

    # DataLoader handles batching and shuffling.
    # drop_last=True: if the final batch has only 1 sample, BatchNorm1d
    # cannot compute a variance over a single observation and throws a
    # RuntimeError. Dropping the remainder (at most 31 samples, <0.5%
    # of training data) prevents this crash with negligible data loss.
    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_tr_t, y_tr_t),
        batch_size=ANN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        drop_last=True,
    )

    # Initialise model, loss function, and optimiser
    model = CariesANN(
        input_dim=X_tr.shape[1],
        output_dim=n_classes,
    )
    criterion = nn.CrossEntropyLoss(
        weight=torch_class_weights(y_tr, n_classes),
    )
    # Adam: adaptive learning rate optimiser. Maintains per-parameter
    # learning rates, making it robust to NHANES features that span
    # very different scales even after standardisation.
    # weight_decay=ANN_WEIGHT_DECAY: L2 penalty added to the loss,
    # shrinking weights toward zero at each update step.
    optimiser = optim.Adam(
        model.parameters(),
        lr=ANN_LEARNING_RATE,
        weight_decay=ANN_WEIGHT_DECAY,
    )

    history = []
    for epoch in range(1, ANN_EPOCHS + 1):
        # --- Training phase ---
        # model.train() enables Dropout and BatchNorm's training mode
        # (uses batch statistics for normalisation).
        model.train()
        batch_losses = []
        for X_batch, y_batch in loader:
            optimiser.zero_grad()          # clear gradients from last step
            logits = model(X_batch)        # forward pass
            loss = criterion(logits, y_batch)  # compute loss
            loss.backward()                # backpropagate gradients
            optimiser.step()               # update weights
            batch_losses.append(loss.item())

        # --- Validation phase ---
        # model.eval() disables Dropout and switches BatchNorm to use
        # running statistics (not batch statistics). torch.no_grad()
        # disables gradient computation — inference only, no memory
        # wasted on gradient storage.
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va_t)
            val_loss = criterion(val_logits, y_va_t).item()

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(batch_losses)),
            "validation_loss": val_loss,
        })

    return model, pd.DataFrame(history)


#===============================================================================
# ANN PREDICTION UTILITY
#===============================================================================
def ann_predict(
    model: CariesANN,
    X: np.ndarray,
) -> tuple:
    """Generate class predictions and probabilities from a trained ANN.

    Args:
        model: Trained CariesANN instance.
        X: Scaled feature array, shape (n_samples, n_features).

    Returns:
        Tuple of (predictions, probabilities):
            predictions: Integer class array, shape (n_samples,).
            probabilities: Float array, shape (n_samples, n_classes).
    """
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32))
        # Apply softmax HERE (not in forward) to get valid probabilities
        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    predictions = np.argmax(probabilities, axis=1)
    return predictions, probabilities


#===============================================================================
# ARCHITECTURE SUMMARY
#===============================================================================
# Print parameter count for the binary case
_demo_model = CariesANN(input_dim=438, output_dim=2)
_total_params = sum(p.numel() for p in _demo_model.parameters())
_trainable = sum(
    p.numel() for p in _demo_model.parameters() if p.requires_grad
)
del _demo_model

print("Model architectures defined.")
print(f"\nRandom Forest:")
print(f"  Trees             : {RF_N_ESTIMATORS}")
print(f"  Class weighting   : {RF_CLASS_WEIGHT}")
print(f"  Feature candidates: sqrt(438) = {int(438**0.5)} per split")
print(f"\nANN (binary case, 438 inputs):")
print(f"  Architecture      : 438 → 128 → 64 → 2")
print(f"  Total parameters  : {_total_params:,}")
print(f"  Trainable params  : {_trainable:,}")
print(f"  Dropout rate      : {ANN_DROPOUT}")
print(f"  Weight decay      : {ANN_WEIGHT_DECAY}")
print(f"\nChildren stratum warning:")
print(f"  Training samples per fold : ~736")
print(f"  Parameters per sample     : {_total_params/736:.0f}:1")
print(f"  High ratio — dropout and weight decay are critical here.")
print("\nProceed to Chapter 7.")

Model architectures defined.

Random Forest:
  Trees             : 200
  Class weighting   : balanced
  Feature candidates: sqrt(438) = 20 per split

ANN (binary case, 438 inputs):
  Architecture      : 438 → 128 → 64 → 2
  Total parameters  : 64,962
  Trainable params  : 64,962
  Dropout rate      : 0.3
  Weight decay      : 0.0001

Children stratum warning:
  Training samples per fold : ~736
  Parameters per sample     : 88:1
  High ratio — dropout and weight decay are critical here.

Proceed to Chapter 7.


In [8]:
#===============================================================================
# CHAPTER 7: EVALUATION AND STATISTICAL COMPARISON UTILITIES
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Define all metric functions and the statistical test used to compare
# models. One metric function is used identically for both models — this
# guarantees the comparison is fair and the results are directly comparable.
#
# WHY THESE FOUR METRICS?
# No single metric tells the full clinical story for caries prediction.
#
# AUC-ROC (Area Under the Receiver Operating Characteristic Curve):
#   The probability that the model ranks a randomly chosen caries-positive
#   patient higher than a randomly chosen caries-negative patient.
#   AUC = 0.5 is chance; AUC = 1.0 is perfect. Critically, AUC is
#   THRESHOLD-INDEPENDENT — it evaluates the model across all possible
#   decision thresholds simultaneously. This makes it the primary metric
#   for comparing models regardless of class imbalance, and the standard
#   in the caries ML literature (Pang et al. 2021; Tirkkonen et al. 2024).
#
# WEIGHTED F1:
#   Harmonic mean of precision and recall, weighted by class size.
#   Captures the trade-off between false positives (over-diagnosing caries)
#   and false negatives (missing caries). More informative than accuracy
#   under class imbalance — a model predicting "no caries" for everyone
#   achieves 62.3% accuracy but F1 = 0 for the caries class.
#
# SENSITIVITY (Recall for the positive class):
#   Proportion of caries-positive patients correctly identified.
#   The clinically critical metric — a missed caries case leads to
#   untreated disease progression. High sensitivity is the priority in
#   a screening context.
#
# SPECIFICITY:
#   Proportion of caries-negative patients correctly identified.
#   Balances sensitivity — if sensitivity is maximised by predicting
#   caries for everyone, specificity collapses to zero. The two together
#   characterise the clinical utility of the model.
#
# THE NADEAU-BENGIO CORRECTED T-TEST:
# Standard paired t-tests assume independent observations. In k-fold
# cross-validation the observations are NOT independent — consecutive
# training folds share approximately 80% of their patients. This
# violates the independence assumption, making the ordinary t-test
# anti-conservative (inflated Type I error — falsely declaring
# significance). Nadeau & Bengio (2003) derived a correction factor
# that accounts for the overlap between folds:
#
#   Correction = (1/k) + (n_test / n_train)
#
# The corrected standard error is:
#   SE_corrected = sqrt(Correction * var(fold_differences))
#
# And the test statistic is:
#   t = mean(fold_differences) / SE_corrected
#
# With df = k-1 degrees of freedom. This is more conservative than the
# standard test — it requires larger differences to declare significance,
# which is appropriate given our small k=5. A non-significant result does
# NOT prove equivalence; it reflects limited power with 5 folds.
#
# Reference: Nadeau C, Bengio Y (2003). Inference for the generalization
# error. Machine Learning, 52(3), 239-281.
# ============================================================================

def align_proba(
    proba: np.ndarray,
    model_classes: np.ndarray,
    expected_classes: list,
) -> np.ndarray:
    """Reorder probability columns to a fixed canonical class order.

    sklearn's predict_proba orders columns by the classes actually seen
    in the training fold. If a rare class (e.g. severe caries in a small
    stratum) is absent from a fold's training set, its column is missing.
    This function forces a consistent layout so metrics and ROC curves
    are always comparable across folds and between models.

    Args:
        proba: Raw probability matrix from predict_proba.
        model_classes: Classes the model was actually trained on.
        expected_classes: Full canonical class list, e.g. [0, 1, 2].

    Returns:
        Aligned probability matrix of shape (n, len(expected_classes)).
    """
    aligned = np.zeros(
        (proba.shape[0], len(expected_classes)), dtype=float
    )
    for src_idx, cls in enumerate(model_classes):
        if int(cls) in expected_classes:
            tgt_idx = expected_classes.index(int(cls))
            aligned[:, tgt_idx] = proba[:, src_idx]
    return aligned


def multiclass_specificity(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    classes: list,
) -> float:
    """Support-weighted one-vs-rest specificity for multiclass outcomes.

    Computes specificity for each class (treating it as positive vs all
    others) then takes a support-weighted average — consistent with how
    sklearn computes weighted metrics.

    Args:
        y_true: True integer labels.
        y_pred: Predicted integer labels.
        classes: All class labels, e.g. [0, 1, 2].

    Returns:
        Weighted specificity scalar, or NaN if undefined.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    specs, weights = [], []
    for c in classes:
        is_c = (y_true == c)
        pred_c = (y_pred == c)
        tn = int(np.sum(~is_c & ~pred_c))
        fp = int(np.sum(~is_c & pred_c))
        support = int(np.sum(is_c))
        if (tn + fp) > 0 and support > 0:
            specs.append(tn / (tn + fp))
            weights.append(support)
    if not specs:
        return np.nan
    return float(np.average(specs, weights=weights))


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
    classes: list,
) -> dict:
    """Compute AUC-ROC, weighted F1, sensitivity, and specificity.

    Handles both binary (len(classes)==2) and multiclass cases with
    identical calling convention — one function for both models and
    both outcomes.

    Args:
        y_true: True integer labels.
        y_pred: Predicted integer labels.
        y_proba: Aligned class probability matrix.
        classes: Canonical class labels, e.g. [0, 1] or [0, 1, 2].

    Returns:
        Dictionary with keys: auc_roc, f1_weighted,
        sensitivity, specificity. Values are NaN where undefined.
    """
    n_classes = len(classes)

    # AUC-ROC
    try:
        if n_classes == 2:
            auc = roc_auc_score(y_true, y_proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, y_proba,
                labels=classes,
                multi_class="ovr",
                average="weighted",
            )
    except ValueError:
        auc = np.nan

    # Weighted F1
    f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0,
    )

    # Sensitivity and specificity
    if n_classes == 2:
        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=classes
        ).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    else:
        sensitivity = recall_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0,
        )
        specificity = multiclass_specificity(y_true, y_pred, classes)

    return {
        "auc_roc": auc,
        "f1_weighted": f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
    }


def nadeau_bengio_ttest(
    values_a: np.ndarray,
    values_b: np.ndarray,
    n_train: int,
    n_test: int,
) -> tuple:
    """Nadeau-Bengio (2003) corrected resampled paired t-test.

    Corrects for the non-independence of k-fold CV observations by
    inflating the standard error via the factor (1/k + n_test/n_train).
    This makes the test more conservative than a standard paired t-test,
    which is appropriate given k=5 gives very limited power.

    Formula:
        d = values_a - values_b  (per-fold differences)
        SE = sqrt((1/k + n_test/n_train) * var(d))
        t = mean(d) / SE
        p = 2 * P(T > |t|) with df = k-1

    NOTE: There is NO additional division by k inside the sqrt. The
    (1/k) term in the correction already replaces the standard error's
    usual (1/k) term. Dividing again would make the test doubly
    conservative — a common implementation error.

    Args:
        values_a: Per-fold metric values for model A (length k).
        values_b: Per-fold metric values for model B (length k).
        n_train: Training set size per fold (for correction factor).
        n_test: Validation set size per fold (for correction factor).

    Returns:
        Tuple of (t_statistic, p_value). Returns (NaN, NaN) if the
        test cannot be computed (fewer than 2 finite differences).
    """
    a = np.asarray(values_a, dtype=float)
    b = np.asarray(values_b, dtype=float)
    d = a - b
    d = d[np.isfinite(d)]   # remove NaN folds gracefully
    k = len(d)

    if k < 2:
        return np.nan, np.nan

    mean_d = d.mean()
    var_d = d.var(ddof=1)   # unbiased sample variance

    if var_d == 0:
        # All fold differences are identical
        if np.isclose(mean_d, 0.0):
            return 0.0, 1.0   # identical performance
        return np.inf, 0.0    # consistent non-zero difference

    # Correction factor: replaces 1/k in the standard SE formula
    correction = (1.0 / k) + (n_test / n_train)
    se_corrected = np.sqrt(correction * var_d)
    t_stat = mean_d / se_corrected
    p_val = float(2.0 * stats.t.sf(np.abs(t_stat), df=k - 1))

    return float(t_stat), p_val


print("Evaluation utilities defined.")
print("\nMetrics computed per fold:")
print("  AUC-ROC      : threshold-independent, primary metric")
print("  Weighted F1  : precision-recall trade-off under imbalance")
print("  Sensitivity  : proportion of caries cases correctly found")
print("  Specificity  : proportion of caries-free correctly identified")
print("\nStatistical test:")
print("  Nadeau-Bengio corrected paired t-test (2003)")
print("  Accounts for non-independence of k-fold CV folds")
print("  More conservative than standard paired t-test")
print("  With k=5: limited power — non-significant ≠ equivalent")
print("\nProceed to Chapter 8.")

Evaluation utilities defined.

Metrics computed per fold:
  AUC-ROC      : threshold-independent, primary metric
  Weighted F1  : precision-recall trade-off under imbalance
  Sensitivity  : proportion of caries cases correctly found
  Specificity  : proportion of caries-free correctly identified

Statistical test:
  Nadeau-Bengio corrected paired t-test (2003)
  Accounts for non-independence of k-fold CV folds
  More conservative than standard paired t-test
  With k=5: limited power — non-significant ≠ equivalent

Proceed to Chapter 8.


In [9]:
#===============================================================================
# CHAPTER 8: UNIFIED EXPERIMENT ENGINE
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Define the function that runs one complete 5-fold cross-validation
# experiment for a single (strategy, outcome, cohort) combination.
# Within each fold it: imputes once (Chapter 5), trains both models on
# identical imputed data (Chapter 6), evaluates both with identical
# metrics (Chapter 7), and stores genuine out-of-fold predictions for
# real ROC curves later — no approximations.
#
# THREE EXPERIMENTAL STRATEGIES:
# This study tests three ways of handling age in the prediction task,
# in increasing order of clinical sophistication:
#
# 1. POOLED BASELINE:
#    All 8,186 patients in one model. Age is excluded as a feature.
#    This is how most published caries ML models are built (Pang et al.
#    2021; Tirkkonen et al. 2024). It is the benchmark to beat.
#    Weakness: forces one decision boundary to represent children at
#    14% prevalence and seniors at 55% simultaneously.
#
# 2. AGE-INCLUSIVE POOLED:
#    All 8,186 patients in one model, but RIDAGEYR is included as a
#    feature. The model can learn the age-caries relationship but must
#    do so implicitly within a single boundary. An intermediate step
#    between pooled and stratified.
#
# 3. AGE-STRATIFIED:
#    Separate model trained and evaluated for each of Orlenko's four
#    age strata. Each model learns the caries pattern specific to its
#    population. This directly tests Orlenko's finding that children
#    and seniors have distinct aetiological signatures — if stratification
#    helps, AUC should improve over the pooled baseline, particularly
#    in the Children and Seniors strata.
#
# WHAT IS STORED PER FOLD:
# - Per-fold metrics for both models (for Nadeau-Bengio test)
# - Genuine out-of-fold predictions and probabilities (for real ROC
#   curves and confusion matrices — not approximations)
# - Random Forest feature importances (averaged across folds)
# - ANN loss history (for convergence plots)
#
# DATA LEAKAGE EXCLUSIONS (_NEVER_FEATURES):
# outcome_s  — the raw score both outcomes are derived from
# outcome_b  — a perfect copy of caries_binary (AUC = 1.0 if included,
#              verified in Chapter 3)
# caries_binary, caries_severity — the prediction targets themselves
# age_stratum — a string label derived from RIDAGEYR, not a raw feature
#===============================================================================

#===============================================================================
# LEAKAGE GUARD — _NEVER_FEATURES
#===============================================================================
# outcome_b is included explicitly following the leakage verification in
# Chapter 3. Any column in this list is silently excluded from all feature
# sets regardless of strategy. Adding a column here is the one-line fix
# for any future leakage discovered in the dataset.
_NEVER_FEATURES = [
    "outcome_s",       # raw score — both targets derived from this
    "outcome_b",       # perfect copy of caries_binary — verified Ch. 3
    "caries_binary",   # primary prediction target
    "caries_severity", # secondary prediction target
    "age_stratum",     # string label derived from RIDAGEYR
]

# Belt-and-braces leakage check: confirm no remaining column perfectly
# predicts the binary outcome. Runs once here, before any model trains.
_numeric_cols = [
    c for c in df.columns
    if c not in _NEVER_FEATURES
    and df[c].dtype in [float, int, "float64"]
]
_leaky = [
    c for c in _numeric_cols
    if abs(df[c].corr(df["caries_binary"])) > 0.99
]
assert not _leaky, (
    f"Potential leakage detected — columns correlate > 0.99 with "
    f"caries_binary: {_leaky}. Add them to _NEVER_FEATURES."
)
print(f"Leakage guard passed. {len(_leaky)} suspicious columns found.")

#===============================================================================
# EXPERIMENT STRATEGIES
#===============================================================================
STRATEGIES = [
    {
        "id": "pooled_baseline",
        "label": "Pooled baseline",
        "include_age": False,
        "stratified": False,
    },
    {
        "id": "age_inclusive",
        "label": "Age-inclusive pooled",
        "include_age": True,
        "stratified": False,
    },
    {
        "id": "age_stratified",
        "label": "Age-stratified",
        "include_age": False,
        "stratified": True,
    },
]

# Outcomes to evaluate
OUTCOMES = {"Binary": "caries_binary"}
if RUN_MULTICLASS:
    OUTCOMES["Multiclass"] = "caries_severity"

# Canonical class labels per outcome — used for metric alignment
CLASS_LABELS = {
    "caries_binary": [0, 1],
    "caries_severity": [0, 1, 2],
}


def feature_columns(include_age: bool, data: pd.DataFrame) -> list:
    """Return predictor column names, excluding leakage columns.

    Columns entirely absent in the provided data are also excluded —
    they carry no predictive information for that population and would
    only add noise to the imputer.

    Args:
        include_age: If True, RIDAGEYR is included as a feature.
                     If False, age is excluded (pooled baseline uses
                     this to prevent implicit age stratification).
        data: The cohort DataFrame — used to detect absent columns.

    Returns:
        List of column names safe to use as model inputs for this
        cohort.
    """
    excluded = list(_NEVER_FEATURES)
    if not include_age:
        excluded.append("RIDAGEYR")

    # Drop columns entirely absent in this cohort — they carry no
    # predictive information for this population and inflate the
    # imputer's workload unnecessarily. This is documented in Methods:
    # "Variables entirely absent within a stratum were excluded from
    # that stratum's feature set prior to modelling."
    return [
        c for c in data.columns
        if c not in excluded
        and data[c].notna().any()
    ]


def run_experiment(
    data: pd.DataFrame,
    strategy: dict,
    outcome_name: str,
    outcome_col: str,
    cohort: str,
) -> dict:
    """Run a complete 5-fold experiment for one configuration.

    Imputes once per fold and trains both models on the same imputed
    split. Stores genuine out-of-fold predictions — not approximations
    — for post-hoc ROC curve construction and confusion matrices.

    Args:
        data: Full df or one age-stratum subset.
        strategy: One entry from STRATEGIES.
        outcome_name: 'Binary' or 'Multiclass'.
        outcome_col: 'caries_binary' or 'caries_severity'.
        cohort: 'All' for pooled strategies, stratum name otherwise.

    Returns:
        Dictionary containing per-fold metrics, pooled out-of-fold
        predictions, averaged RF importances, and ANN loss history.
    """
    classes = CLASS_LABELS[outcome_col]

    # feature_columns now handles absent-column exclusion per cohort,
    # replacing the per-fold warning and zero-fill for structural
    # missingness. The zero-fill in preprocess_fold still handles
    # fold-level absence (rare edge case within a stratum fold).
    feat_cols = feature_columns(strategy["include_age"], data)

    X = data[feat_cols].reset_index(drop=True)
    y = data[outcome_col].astype(int).reset_index(drop=True).values

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED,
    )

    rf_fold_metrics, ann_fold_metrics = [], []
    rf_importance_acc = []
    ann_history_acc = []

    # Out-of-fold storage — genuine predictions accumulated across folds.
    # These are used directly for ROC curves and confusion matrices,
    # never recomputed or approximated.
    oof = {
        "Random Forest": {"y_true": [], "y_proba": [], "y_pred": []},
        "ANN":           {"y_true": [], "y_proba": [], "y_pred": []},
    }

    n_train_rep, n_test_rep = 0, 0

    for fold, (tr_idx, va_idx) in enumerate(
        skf.split(X, y), start=1
    ):
        # Unique seed per fold and stratum — reproducible but varied
        # so different folds use different imputer random states.
        if cohort in AGE_STRATA:
            fold_seed = (
                SEED + fold + AGE_STRATA.index(cohort) * 13
            )
        else:
            fold_seed = SEED + fold

        X_tr_df = X.iloc[tr_idx]
        X_va_df = X.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]
        n_train_rep = len(tr_idx)
        n_test_rep = len(va_idx)

        # ---- Impute ONCE — both models get identical imputed data ----
        X_tr_rf, X_va_rf, X_tr_ann, X_va_ann = preprocess_fold(
            X_tr_df, X_va_df, seed=fold_seed
        )

        # ---- Random Forest ----
        rf = make_random_forest(seed=fold_seed)
        rf.fit(X_tr_rf, y_tr)
        rf_proba_raw = rf.predict_proba(X_va_rf)
        rf_proba = align_proba(rf_proba_raw, rf.classes_, classes)
        rf_pred = rf.predict(X_va_rf)

        rf_fold_metrics.append(
            compute_metrics(y_va, rf_pred, rf_proba, classes)
        )
        rf_importance_acc.append(
            pd.Series(rf.feature_importances_, index=feat_cols)
        )
        oof["Random Forest"]["y_true"].append(y_va)
        oof["Random Forest"]["y_proba"].append(rf_proba)
        oof["Random Forest"]["y_pred"].append(rf_pred)

        # ---- ANN (same imputed fold, additionally standardised) ----
        ann_model, ann_hist = train_ann(
            X_tr_ann, y_tr,
            X_va_ann, y_va,
            n_classes=len(classes),
            seed=fold_seed,
        )
        ann_pred, ann_proba_raw = ann_predict(ann_model, X_va_ann)
        ann_proba = align_proba(
            ann_proba_raw,
            np.arange(len(classes)),
            classes,
        )
        ann_fold_metrics.append(
            compute_metrics(y_va, ann_pred, ann_proba, classes)
        )
        ann_hist["fold"] = fold
        ann_history_acc.append(ann_hist)
        oof["ANN"]["y_true"].append(y_va)
        oof["ANN"]["y_proba"].append(ann_proba)
        oof["ANN"]["y_pred"].append(ann_pred)

    # Concatenate fold arrays into single population-level vectors
    for model_name in ("Random Forest", "ANN"):
        oof[model_name]["y_true"] = np.concatenate(
            oof[model_name]["y_true"]
        )
        oof[model_name]["y_proba"] = np.concatenate(
            oof[model_name]["y_proba"]
        )
        oof[model_name]["y_pred"] = np.concatenate(
            oof[model_name]["y_pred"]
        )

    return {
        "strategy":         strategy["label"],
        "outcome":          outcome_name,
        "outcome_col":      outcome_col,
        "cohort":           cohort,
        "rf_fold_metrics":  rf_fold_metrics,
        "ann_fold_metrics": ann_fold_metrics,
        "rf_importances":   (
            pd.concat(rf_importance_acc, axis=1).mean(axis=1)
        ),
        "ann_loss_history": pd.concat(
            ann_history_acc, ignore_index=True
        ),
        "oof":     oof,
        "n_train": n_train_rep,
        "n_test":  n_test_rep,
    }


def get_experiment(
    strategy: str,
    outcome: str,
    cohort: str,
) -> dict:
    """Fetch a stored experiment result by its identifiers.

    Args:
        strategy: Strategy label string.
        outcome: 'Binary' or 'Multiclass'.
        cohort: 'All' or an age stratum name.

    Returns:
        Matching experiment dict, or None if not found.
    """
    for exp in all_experiments:
        if (
            exp["strategy"] == strategy
            and exp["outcome"] == outcome
            and exp["cohort"] == cohort
        ):
            return exp
    return None


print("Experiment engine defined.")
print(f"\nFeature set sizes (full dataset):")
print(f"  Without age : {len(feature_columns(False, df))} features")
print(f"  With age    : {len(feature_columns(True, df))} features")
print(f"\nNote: stratified cohorts will have fewer features")
print(f"  (columns entirely absent in a stratum are excluded)")
print(f"\nStrategies registered: {len(STRATEGIES)}")
for s in STRATEGIES:
    print(f"  {s['label']}")
print(f"\nOutcomes registered: {len(OUTCOMES)}")
for name in OUTCOMES:
    print(f"  {name}")
print(f"\nLeakage guard: outcome_b confirmed excluded.")
print("\nProceed to Chapter 9.")

Leakage guard passed. 0 suspicious columns found.
Experiment engine defined.

Feature set sizes (full dataset):
  Without age : 437 features
  With age    : 438 features

Note: stratified cohorts will have fewer features
  (columns entirely absent in a stratum are excluded)

Strategies registered: 3
  Pooled baseline
  Age-inclusive pooled
  Age-stratified

Outcomes registered: 2
  Binary
  Multiclass

Leakage guard: outcome_b confirmed excluded.

Proceed to Chapter 9.


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [10]:
#===============================================================================
# CHAPTER 9: EXECUTE ALL EXPERIMENTS AND CONSOLIDATE
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Run every (strategy × outcome) combination and consolidate results into
# tidy tables. For pooled strategies, one experiment runs on all 8,186
# patients. For the stratified strategy, four separate experiments run —
# one per age stratum — each with its own imputation, its own models,
# and its own cross-validation folds.
#
# OUTPUT:
# all_experiments : list of raw experiment dicts (one per configuration)
# results_df      : tidy long-format table, one row per (model, fold)
# summary_df      : mean and SD per configuration, for reporting
#===============================================================================

#===============================================================================
# INITIALISE RESULT STORES
#===============================================================================
all_experiments = []
fold_records = []

#===============================================================================
# EXECUTE
#===============================================================================
start_time = time.time()

for outcome_name, outcome_col in OUTCOMES.items():
    print(f"\n{'='*60}")
    print(f"OUTCOME: {outcome_name}  ({outcome_col})")
    print(f"{'='*60}")

    for strategy in STRATEGIES:

        if strategy["stratified"]:
            # Age-stratified: one separate experiment per stratum
            print(f"\nStrategy: {strategy['label']}")
            for stratum in AGE_STRATA:
                sub = df[df["age_stratum"] == stratum].copy()
                print(
                    f"  {stratum:<9} (n={len(sub):,}) ... ",
                    end="", flush=True,
                )
                exp = run_experiment(
                    sub, strategy,
                    outcome_name, outcome_col,
                    cohort=stratum,
                )
                all_experiments.append(exp)

                rf_auc = np.nanmean(
                    [m["auc_roc"] for m in exp["rf_fold_metrics"]]
                )
                ann_auc = np.nanmean(
                    [m["auc_roc"] for m in exp["ann_fold_metrics"]]
                )
                print(f"RF AUC={rf_auc:.3f}  ANN AUC={ann_auc:.3f}")

        else:
            # Pooled strategies: one experiment on the full dataset
            print(
                f"\nStrategy: {strategy['label']} ... ",
                end="", flush=True,
            )
            exp = run_experiment(
                df, strategy,
                outcome_name, outcome_col,
                cohort="All",
            )
            all_experiments.append(exp)

            rf_auc = np.nanmean(
                [m["auc_roc"] for m in exp["rf_fold_metrics"]]
            )
            ann_auc = np.nanmean(
                [m["auc_roc"] for m in exp["ann_fold_metrics"]]
            )
            print(f"RF AUC={rf_auc:.3f}  ANN AUC={ann_auc:.3f}")

elapsed = (time.time() - start_time) / 60.0
print(f"\nAll experiments completed in {elapsed:.1f} minutes.")
print(f"Total experiments stored: {len(all_experiments)}")

#===============================================================================
# BUILD TIDY LONG-FORMAT TABLE
#===============================================================================
# One row per (model, strategy, outcome, cohort, fold).
METRICS = ["auc_roc", "f1_weighted", "sensitivity", "specificity"]

for exp in all_experiments:
    for model_name, metric_key in (
        ("Random Forest", "rf_fold_metrics"),
        ("ANN",           "ann_fold_metrics"),
    ):
        for fold_idx, met in enumerate(exp[metric_key], start=1):
            fold_records.append({
                "model":    model_name,
                "strategy": exp["strategy"],
                "outcome":  exp["outcome"],
                "cohort":   exp["cohort"],
                "fold":     fold_idx,
                "n_train":  exp["n_train"],
                "n_test":   exp["n_test"],
                **met,
            })

results_df = pd.DataFrame(fold_records)

# Save per-fold metrics — the raw evidence behind every reported number
results_path = os.path.join(TABLE_DIR, "all_fold_metrics.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved per-fold metrics: {len(results_df)} rows")
print(f"  Path: {results_path}")

# Expected row count sanity check
expected = (
    (2 + len(AGE_STRATA)) * len(OUTCOMES) * 2 * N_SPLITS
)
if len(results_df) == expected:
    print(f"  Row count: {len(results_df)} ✓ (expected {expected})")
else:
    print(
        f"  WARNING: expected {expected} rows, "
        f"got {len(results_df)}. Check for missing experiments."
    )

#===============================================================================
# BUILD SUMMARY TABLE — mean and SD per configuration
#===============================================================================
summary_rows = []
group_cols = ["model", "strategy", "outcome", "cohort"]

for keys, grp in results_df.groupby(group_cols):
    row = dict(zip(group_cols, keys))
    for m in METRICS:
        vals = grp[m].dropna().values
        row[f"{m}_mean"] = np.nanmean(vals) if len(vals) else np.nan
        row[f"{m}_std"] = (
            np.nanstd(vals, ddof=1) if len(vals) > 1 else np.nan
        )
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(TABLE_DIR, "summary_metrics.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved summary metrics: {len(summary_df)} configurations")

#===============================================================================
# QUICK RESULTS PREVIEW — binary outcome, pooled cohorts only
#===============================================================================
print("\nBinary outcome — mean AUC-ROC by configuration:")
print(f"{'Model':<15} {'Strategy':<22} {'Cohort':<10} {'AUC':>6}")
print("-" * 56)
_preview = (
    summary_df[summary_df["outcome"] == "Binary"]
    .sort_values(["model", "strategy", "cohort"])
)
for _, r in _preview.iterrows():
    print(
        f"{r['model']:<15} {r['strategy']:<22} "
        f"{r['cohort']:<10} {r['auc_roc_mean']:>6.3f}"
    )

print(f"\nProceed to Chapter 10.")


OUTCOME: Binary  (caries_binary)

Strategy: Pooled baseline ... RF AUC=0.760  ANN AUC=0.709

Strategy: Age-inclusive pooled ... RF AUC=0.762  ANN AUC=0.718

Strategy: Age-stratified
  Children  (n=921) ... RF AUC=0.839  ANN AUC=0.768
  Youth     (n=2,034) ... RF AUC=0.733  ANN AUC=0.720
  Adults    (n=3,942) ... RF AUC=0.665  ANN AUC=0.630
  Seniors   (n=1,289) ... RF AUC=0.756  ANN AUC=0.712

OUTCOME: Multiclass  (caries_severity)

Strategy: Pooled baseline ... RF AUC=0.728  ANN AUC=0.678

Strategy: Age-inclusive pooled ... RF AUC=0.732  ANN AUC=0.679

Strategy: Age-stratified
  Children  (n=921) ... RF AUC=0.824  ANN AUC=0.749
  Youth     (n=2,034) ... RF AUC=0.714  ANN AUC=0.694
  Adults    (n=3,942) ... RF AUC=0.627  ANN AUC=0.607
  Seniors   (n=1,289) ... RF AUC=0.700  ANN AUC=0.672

All experiments completed in 59.9 minutes.
Total experiments stored: 12

Saved per-fold metrics: 120 rows
  Path: /content/drive/MyDrive/DASC512_Assessment2/outputs/tables/all_fold_metrics.csv
  Row 

In [12]:
#===============================================================================
# CHAPTER 10: STATISTICAL COMPARISON — NADEAU-BENGIO CORRECTED
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Answer three scientific questions with appropriate statistics:
#
# Q1. Does age stratification improve prediction over the pooled baseline?
#     This is the core question — the direct test of Orlenko's finding
#     that age-driven heterogeneity exists and matters predictively.
#
# Q2. Does Random Forest outperform the ANN on this dataset?
#     Architecture comparison — shallow vs deep learning for tabular
#     clinical data.
#
# Q3. Does outcome granularity (binary vs multiclass) affect performance?
#     Secondary question — does preserving severity information help
#     or hurt the models?
#
# WHY THE NADEAU-BENGIO CORRECTION IS MANDATORY HERE:
# Each comparison uses per-fold metric values as paired observations.
# With k=5, consecutive folds share ~80% of their training patients —
# violating the independence assumption of a standard paired t-test.
# The Nadeau-Bengio correction inflates the standard error to account
# for this overlap. Without it, we would overstate statistical confidence
# in differences that may reflect fold overlap rather than true effects.
#===============================================================================

def pooled_fold_values(
    model: str,
    strategy: str,
    outcome: str,
    metric: str,
) -> np.ndarray:
    """Per-fold metric values for a pooled (cohort='All') configuration.

    Args:
        model: 'Random Forest' or 'ANN'.
        strategy: Strategy label string.
        outcome: 'Binary' or 'Multiclass'.
        metric: One of the METRICS column names.

    Returns:
        Array of per-fold values, sorted by fold number.
    """
    mask = (
        (results_df["model"] == model)
        & (results_df["strategy"] == strategy)
        & (results_df["outcome"] == outcome)
        & (results_df["cohort"] == "All")
    )
    return results_df.loc[mask].sort_values("fold")[metric].values


def stratified_fold_values(
    model: str,
    outcome: str,
    metric: str,
) -> np.ndarray:
    """Per-fold values for the stratified strategy, averaged across strata.

    For each fold, takes the mean metric across the four age strata.
    This gives 5 paired fold values for comparison against the pooled
    baseline's 5 fold values.

    NOTE: This is an unweighted mean — Children (n=921) contributes
    equally to Adults (n=3,942). A weighted mean would give more
    influence to larger strata. We use unweighted for consistency with
    how the Nadeau-Bengio test is applied, and note this as a limitation.

    Args:
        model: 'Random Forest' or 'ANN'.
        outcome: 'Binary' or 'Multiclass'.
        metric: One of the METRICS column names.

    Returns:
        Array of 5 fold-averaged values.
    """
    mask = (
        (results_df["model"] == model)
        & (results_df["strategy"] == "Age-stratified")
        & (results_df["outcome"] == outcome)
    )
    return (
        results_df.loc[mask]
        .groupby("fold")[metric]
        .mean()
        .sort_index()
        .values
    )


def representative_split_sizes(outcome: str) -> tuple:
    """Pooled-baseline fold sizes for the Nadeau-Bengio correction.

    Uses pooled baseline sizes as the reference for all comparisons.
    For stratified vs pooled comparisons this is an approximation —
    stratified folds are smaller — but the correction factor is a
    scalar multiplier and small size differences have minimal impact.

    Args:
        outcome: 'Binary' or 'Multiclass'.

    Returns:
        Tuple of (n_train, n_test) from the pooled baseline.
    """
    mask = (
        (results_df["strategy"] == "Pooled baseline")
        & (results_df["outcome"] == outcome)
        & (results_df["cohort"] == "All")
    )
    row = results_df.loc[mask].iloc[0]
    return int(row["n_train"]), int(row["n_test"])


#===============================================================================
# RUN ALL COMPARISONS
#===============================================================================
comparison_rows = []

# ---- Q1: Pooled baseline vs Age-stratified ----
for model in ("Random Forest", "ANN"):
    for outcome in OUTCOMES:
        n_tr, n_te = representative_split_sizes(outcome)
        for metric in METRICS:
            a = pooled_fold_values(
                model, "Pooled baseline", outcome, metric
            )
            b = stratified_fold_values(model, outcome, metric)
            if len(a) == 0 or len(b) == 0:
                continue
            t, p = nadeau_bengio_ttest(b, a, n_tr, n_te)
            comparison_rows.append({
                "comparison":      "Pooled vs Age-stratified",
                "model":           model,
                "outcome":         outcome,
                "metric":          metric,
                "mean_a":          np.nanmean(a),
                "mean_b":          np.nanmean(b),
                "difference":      np.nanmean(b) - np.nanmean(a),
                "t_statistic":     t,
                "p_value":         p,
                "significant_05":  (
                    bool(p < 0.05) if np.isfinite(p) else False
                ),
            })

# ---- Q2: Random Forest vs ANN ----
for strategy in ("Pooled baseline", "Age-inclusive pooled"):
    for outcome in OUTCOMES:
        n_tr, n_te = representative_split_sizes(outcome)
        for metric in METRICS:
            a = pooled_fold_values(
                "Random Forest", strategy, outcome, metric
            )
            b = pooled_fold_values(
                "ANN", strategy, outcome, metric
            )
            if len(a) == 0 or len(b) == 0:
                continue
            t, p = nadeau_bengio_ttest(a, b, n_tr, n_te)
            comparison_rows.append({
                "comparison":      f"RF vs ANN ({strategy})",
                "model":           "RF vs ANN",
                "outcome":         outcome,
                "metric":          metric,
                "mean_a":          np.nanmean(a),
                "mean_b":          np.nanmean(b),
                "difference":      np.nanmean(a) - np.nanmean(b),
                "t_statistic":     t,
                "p_value":         p,
                "significant_05":  (
                    bool(p < 0.05) if np.isfinite(p) else False
                ),
            })

# ---- Q3: Binary vs Multiclass (if both outcomes were run) ----
if RUN_MULTICLASS:
    for model in ("Random Forest", "ANN"):
        for strategy in ("Pooled baseline", "Age-inclusive pooled"):
            n_tr, n_te = representative_split_sizes("Binary")
            for metric in METRICS:
                a = pooled_fold_values(
                    model, strategy, "Binary", metric
                )
                b = pooled_fold_values(
                    model, strategy, "Multiclass", metric
                )
                if len(a) == 0 or len(b) == 0:
                    continue
                t, p = nadeau_bengio_ttest(a, b, n_tr, n_te)
                comparison_rows.append({
                    "comparison":  (
                        f"Binary vs Multiclass "
                        f"({model}, {strategy})"
                    ),
                    "model":       model,
                    "outcome":     "Binary vs Multiclass",
                    "metric":      metric,
                    "mean_a":      np.nanmean(a),
                    "mean_b":      np.nanmean(b),
                    "difference":  np.nanmean(a) - np.nanmean(b),
                    "t_statistic": t,
                    "p_value":     p,
                    "significant_05": (
                        bool(p < 0.05) if np.isfinite(p) else False
                    ),
                })

comparison_df = pd.DataFrame(comparison_rows)
comp_path = os.path.join(TABLE_DIR, "statistical_comparisons.csv")
comparison_df.to_csv(comp_path, index=False)
print(f"Statistical comparisons saved: {len(comparison_df)} tests")

#===============================================================================
# PRINT CORE RESULTS — Q1 and Q2, AUC only
#===============================================================================
print("\n" + "=" * 65)
print("Q1: Pooled baseline vs Age-stratified — AUC-ROC")
print("=" * 65)
q1 = comparison_df[
    (comparison_df["comparison"] == "Pooled vs Age-stratified")
    & (comparison_df["metric"] == "auc_roc")
][["model", "outcome", "mean_a", "mean_b",
   "difference", "p_value", "significant_05"]]
print(q1.to_string(index=False))

print("\n" + "=" * 65)
print("Q2: Random Forest vs ANN — AUC-ROC (pooled baseline)")
print("=" * 65)
q2 = comparison_df[
    (comparison_df["comparison"] == "RF vs ANN (Pooled baseline)")
    & (comparison_df["metric"] == "auc_roc")
][["model", "outcome", "mean_a", "mean_b",
   "difference", "p_value", "significant_05"]]
print(q2.to_string(index=False))

Statistical comparisons saved: 48 tests

Q1: Pooled baseline vs Age-stratified — AUC-ROC
        model    outcome   mean_a   mean_b  difference  p_value  significant_05
Random Forest     Binary 0.759860 0.748091   -0.011770 0.318409           False
Random Forest Multiclass 0.728312 0.716231   -0.012081 0.351496           False
          ANN     Binary 0.709128 0.707498   -0.001630 0.911998           False
          ANN Multiclass 0.677575 0.680295    0.002720 0.823330           False

Q2: Random Forest vs ANN — AUC-ROC (pooled baseline)
    model    outcome   mean_a   mean_b  difference  p_value  significant_05
RF vs ANN     Binary 0.759860 0.709128    0.050733 0.006800            True
RF vs ANN Multiclass 0.728312 0.677575    0.050738 0.030242            True


In [13]:
#===============================================================================
# CHAPTER 11: VISUALISATIONS
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Produce all eight submission figures from the genuine out-of-fold
# predictions stored in Chapter 9. No curve is approximated or simulated.
# No model is retrained. Every figure is built from what actually happened
# during cross-validation.
#
# FIGURE PLAN:
# Fig 1 — Caries prevalence by age stratum (produced in Chapter 4)
# Fig 2 — Random Forest ROC curves, three strategies, binary
# Fig 3 — ANN ROC curves, three strategies, binary
# Fig 4 — Metric comparison bar chart (binary and multiclass)
# Fig 5 — ANN training vs validation loss curves
# Fig 6 — Confusion matrices, best model per stratum
# Fig 7 — Random Forest feature importances per stratum
# Fig 8 — Summary results table
#
# NOTE ON FIGURE TITLES:
# fig.suptitle() has been removed from all figures.
# Figure numbers and titles are carried exclusively by report captions.
# This is standard academic practice — captions live in the document,
# not embedded in the image file.
#===============================================================================

def pooled_oof_for_strategy(
    model: str,
    strategy: str,
    outcome: str,
) -> tuple:
    """Get pooled out-of-fold predictions for one strategy.

    For pooled strategies: returns the stored OOF directly.
    For the stratified strategy: concatenates OOF from all four
    strata to give one population-level result — population-weighted
    and genuine. This ensures Figures 2, 3, and 4 are all internally
    consistent — the AUC shown in the ROC curve legend matches the
    bar height in the metric comparison chart.

    Args:
        model: 'Random Forest' or 'ANN'.
        strategy: Strategy label string.
        outcome: 'Binary' or 'Multiclass'.

    Returns:
        Tuple of (y_true, positive_class_proba) for binary,
        or (y_true, full_proba_matrix) — caller decides usage.
        Returns (None, None) if experiment not found.
    """
    if strategy == "Age-stratified":
        ys, ss = [], []
        for stratum in AGE_STRATA:
            exp = get_experiment(strategy, outcome, stratum)
            if exp is None:
                continue
            ys.append(exp["oof"][model]["y_true"])
            ss.append(exp["oof"][model]["y_proba"][:, 1])
        if not ys:
            return None, None
        return np.concatenate(ys), np.concatenate(ss)
    else:
        exp = get_experiment(strategy, outcome, "All")
        if exp is None:
            return None, None
        return (
            exp["oof"][model]["y_true"],
            exp["oof"][model]["y_proba"][:, 1],
        )


#===============================================================================
# FIGURE 2 & 3: ROC CURVES — one figure per model, binary outcome
#===============================================================================
def plot_roc_curves(model: str, fname: str) -> str:
    """Plot binary ROC curves for all three strategies.

    Each curve is drawn from genuine concatenated out-of-fold
    predictions — not approximations. The AUC shown in the legend
    is recomputed from the same predictions for consistency.
    Figure title removed — carried by report caption only.

    Args:
        model: 'Random Forest' or 'ANN'.
        fname: Output filename.

    Returns:
        Full path of the saved figure.
    """
    fig, ax = plt.subplots(figsize=(7.5, 6.5))

    for strat in STRATEGIES:
        y_true, y_score = pooled_oof_for_strategy(
            model, strat["label"], "Binary"
        )
        if y_true is None:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc_val = roc_auc_score(y_true, y_score)
        ax.plot(
            fpr, tpr,
            linewidth=2.4,
            color=PALETTE[strat["label"]],
            label=f"{strat['label']} (AUC = {auc_val:.3f})",
        )

    ax.plot(
        [0, 1], [0, 1],
        linestyle="--", color="grey",
        linewidth=1, label="Chance (AUC = 0.500)",
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("False positive rate (1 − specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.legend(loc="lower right", fontsize=9)

    # Caption embedded for reference — figure number/title in report only
    caption = (
        f"Receiver operating characteristic curves for {model} across "
        f"three age-handling strategies. Curves are constructed from "
        f"genuine out-of-fold predictions accumulated across 5 stratified "
        f"folds. AUC values are recomputed from pooled predictions. "
        f"Chance line (AUC=0.500) shown for reference."
    )
    fig.text(
        0.5, -0.04, caption,
        ha="center", fontsize=7.5, style="italic",
        wrap=True, transform=fig.transFigure,
    )
    fig.tight_layout()
    return save_figure(fig, fname)


FIG2_PATH = plot_roc_curves(
    "Random Forest", "Figure_2_RF_ROC_Curves.png"
)
FIG3_PATH = plot_roc_curves(
    "ANN", "Figure_3_ANN_ROC_Curves.png"
)

#===============================================================================
# FIGURE 4: METRIC COMPARISON BAR CHART
#===============================================================================
# For the stratified strategy, metrics are derived from concatenated OOF
# predictions — consistent with Figures 2 and 3. This ensures the AUC
# bar for "Age-stratified" matches the AUC shown in the ROC curve legend.


def build_stratified_summary_row(
    model: str,
    outcome: str,
) -> dict:
    """Compute metrics from concatenated stratified OOF predictions.

    Handles both binary and multiclass outcomes correctly.
    For binary: uses positive class probability (column 1) for AUC.
    For multiclass: uses the full probability matrix with one-vs-rest
    weighted AUC — not a single column, which would only represent
    one class and produce meaningless results.

    This ensures Figure 4 stratified bars are population-weighted
    and internally consistent with Figures 2 and 3.

    Args:
        model: 'Random Forest' or 'ANN'.
        outcome: 'Binary' or 'Multiclass'.

    Returns:
        Summary row dict with mean metric values, or empty dict
        if no experiments found.
    """
    # Collect genuine OOF predictions across all four strata
    ys, probas, preds = [], [], []
    for stratum in AGE_STRATA:
        exp = get_experiment("Age-stratified", outcome, stratum)
        if exp is None:
            continue
        ys.append(exp["oof"][model]["y_true"])
        probas.append(exp["oof"][model]["y_proba"])
        preds.append(exp["oof"][model]["y_pred"])

    if not ys:
        return {}

    y_true = np.concatenate(ys)
    y_proba = np.concatenate(probas)
    y_pred = np.concatenate(preds)

    classes = CLASS_LABELS[
        "caries_binary" if outcome == "Binary"
        else "caries_severity"
    ]
    n_classes = len(classes)

    # AUC — binary and multiclass require different approaches
    try:
        if n_classes == 2:
            # Binary: probability of positive class (column 1)
            auc = roc_auc_score(y_true, y_proba[:, 1])
        else:
            # Multiclass: one-vs-rest weighted AUC using full matrix.
            # The previous version used y_proba[:, 1] here — column 1
            # is mild caries probability only, not a valid multiclass
            # score. This fix uses the full probability matrix.
            auc = roc_auc_score(
                y_true, y_proba,
                labels=classes,
                multi_class="ovr",
                average="weighted",
            )
    except ValueError:
        auc = np.nan

    # Weighted F1
    f1 = f1_score(
        y_true, y_pred,
        average="weighted",
        zero_division=0,
    )

    # Sensitivity and specificity
    if n_classes == 2:
        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=classes
        ).ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    else:
        # Multiclass: weighted one-vs-rest for both metrics
        sens = recall_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0,
        )
        spec = multiclass_specificity(y_true, y_pred, classes)

    return {
        "model":            model,
        "strategy":         "Age-stratified",
        "outcome":          outcome,
        "cohort":           "All",
        "auc_roc_mean":     auc,
        "auc_roc_std":      np.nan,
        "f1_weighted_mean": f1,
        "f1_weighted_std":  np.nan,
        "sensitivity_mean": sens,
        "sensitivity_std":  np.nan,
        "specificity_mean": spec,
        "specificity_std":  np.nan,
    }


for outcome_name in OUTCOMES:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=False)

    # suptitle removed — figure number and title in report caption only

    # Build display summary for this outcome
    bin_summary = summary_df[
        (summary_df["outcome"] == outcome_name)
        & (summary_df["cohort"] == "All")
    ].copy()

    # Add stratified rows from concatenated OOF — population weighted
    # and consistent with ROC curves. Fix applied here for multiclass.
    for model in ("Random Forest", "ANN"):
        row = build_stratified_summary_row(model, outcome_name)
        if row:
            bin_summary = pd.concat(
                [bin_summary, pd.DataFrame([row])],
                ignore_index=True,
            )

    metric_titles = {
        "auc_roc":       "AUC-ROC",
        "f1_weighted":   "Weighted F1",
        "sensitivity":   "Sensitivity",
        "specificity":   "Specificity",
    }

    for ax, metric in zip(axes.ravel(), METRICS):
        plot_data = []
        for _, r in bin_summary.iterrows():
            plot_data.append({
                "model":    r["model"],
                "strategy": r["strategy"],
                "value":    r[f"{metric}_mean"],
            })
        plot_df = pd.DataFrame(plot_data)

        sns.barplot(
            data=plot_df,
            x="strategy",
            y="value",
            hue="model",
            order=[
                "Pooled baseline",
                "Age-inclusive pooled",
                "Age-stratified",
            ],
            palette={
                "Random Forest": PALETTE["Random Forest"],
                "ANN":           PALETTE["ANN"],
            },
            ax=ax,
        )
        ax.set_title(metric_titles[metric], fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylabel("Mean over 5 folds")
        ax.set_ylim(
            max(0, plot_df["value"].min() - 0.05),
            min(1, plot_df["value"].max() + 0.05),
        )
        ax.tick_params(axis="x", rotation=18)
        ax.legend(title="Model", fontsize=8)

    fig.tight_layout()
    fname = f"Figure_4_{outcome_name}_Metric_Comparison.png"
    save_figure(fig, fname)

#===============================================================================
# FIGURE 5: ANN TRAINING vs VALIDATION LOSS CURVES
#===============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

# suptitle removed — figure number and title in report caption only

for ax, strat in zip(axes, STRATEGIES):
    if strat["stratified"]:
        hists = []
        for s in AGE_STRATA:
            exp = get_experiment(strat["label"], "Binary", s)
            if exp is not None:
                hists.append(exp["ann_loss_history"])
        if not hists:
            continue
        hist = pd.concat(hists, ignore_index=True)
    else:
        exp = get_experiment(strat["label"], "Binary", "All")
        if exp is None:
            continue
        hist = exp["ann_loss_history"]

    mean_curve = (
        hist.groupby("epoch")[["train_loss", "validation_loss"]]
        .mean()
        .reset_index()
    )
    ax.plot(
        mean_curve["epoch"], mean_curve["train_loss"],
        linewidth=2, color=PALETTE[strat["label"]],
        label="Train",
    )
    ax.plot(
        mean_curve["epoch"], mean_curve["validation_loss"],
        linewidth=2, linestyle="--",
        color=PALETTE[strat["label"]],
        label="Validation",
    )
    # Panel title retained — identifies which strategy each subplot shows
    ax.set_title(strat["label"], fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend(loc="upper right", fontsize=8)

caption = (
    "Mean ANN cross-entropy loss per epoch across 5 folds for each "
    "strategy. Divergence between training and validation loss indicates "
    "overfitting. Stratified panel averages loss across all four "
    "age-stratum models."
)
fig.text(
    0.5, -0.04, caption,
    ha="center", fontsize=7.5, style="italic",
    transform=fig.transFigure,
)
fig.tight_layout()
save_figure(fig, "Figure_5_ANN_Loss_Curves.png")

#===============================================================================
# FIGURE 6: CONFUSION MATRICES — best model per stratum
#===============================================================================
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

# suptitle removed — figure number and title in report caption only

for ax, stratum in zip(axes.ravel(), AGE_STRATA):
    exp_rf = get_experiment("Age-stratified", "Binary", stratum)
    if exp_rf is None:
        ax.set_visible(False)
        continue

    rf_auc = np.nanmean(
        [m["auc_roc"] for m in exp_rf["rf_fold_metrics"]]
    )
    ann_auc = np.nanmean(
        [m["auc_roc"] for m in exp_rf["ann_fold_metrics"]]
    )

    if rf_auc >= ann_auc:
        best_model = "Random Forest"
        best_auc = rf_auc
    else:
        best_model = "ANN"
        best_auc = ann_auc

    cm = confusion_matrix(
        exp_rf["oof"][best_model]["y_true"],
        exp_rf["oof"][best_model]["y_pred"],
        labels=[0, 1],
    )
    sns.heatmap(
        cm, annot=True, fmt="d",
        cmap="Blues", cbar=False,
        xticklabels=["Pred: No caries", "Pred: Caries"],
        yticklabels=["True: No caries", "True: Caries"],
        ax=ax,
    )
    n_stratum = len(exp_rf["oof"][best_model]["y_true"])
    # Subplot title retained — identifies stratum, n, best model, AUC
    ax.set_title(
        f"{stratum}  (n={n_stratum:,})  —  "
        f"Best: {best_model}  (AUC {best_auc:.3f})",
        fontweight="bold", fontsize=9,
    )

caption = (
    "Confusion matrices for the best-performing model (Random Forest "
    "or ANN) in each age stratum under the age-stratified strategy. "
    "Values are pooled out-of-fold counts across all 5 folds. "
    "AUC shown is the mean across folds."
)
fig.text(
    0.5, -0.03, caption,
    ha="center", fontsize=7.5, style="italic",
    transform=fig.transFigure,
)
fig.tight_layout()
save_figure(fig, "Figure_6_Confusion_Matrices.png")

#===============================================================================
# FIGURE 7: RANDOM FOREST FEATURE IMPORTANCES PER STRATUM
#===============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# suptitle removed — figure number and title in report caption only

for ax, stratum in zip(axes.ravel(), AGE_STRATA):
    exp = get_experiment("Age-stratified", "Binary", stratum)
    if exp is None:
        ax.set_visible(False)
        continue

    importances = exp["rf_importances"].nlargest(20).sort_values()

    # Map NHANES codes to readable clinical labels — defined in Chapter 1.
    # Variables not in the dictionary fall back to their raw code name.
    readable_labels = [
        NHANES_LABELS.get(idx, idx)
        for idx in importances.index
    ]

    ax.barh(
        range(len(importances)),
        importances.values,
        color=PALETTE["Pooled baseline"],
        edgecolor="white", linewidth=0.5,
    )
    ax.set_yticks(range(len(importances)))
    ax.set_yticklabels(readable_labels, fontsize=8)
    ax.set_xlabel("Mean decrease in Gini impurity")
    # Subplot title retained — identifies stratum and n
    ax.set_title(
        f"{stratum}  "
        f"(n={len(df[df['age_stratum'] == stratum]):,})",
        fontweight="bold",
    )
    ax.axvline(
        importances.values.mean(),
        color="red", linestyle="--",
        linewidth=1, alpha=0.7,
        label="Mean importance",
    )
    ax.legend(fontsize=7)

caption = (
    "Top 20 Random Forest feature importances (mean decrease in Gini "
    "impurity) per Orlenko age stratum, binary caries outcome. "
    "Importances are averaged across 5 folds. NHANES variable codes "
    "are mapped to clinical descriptors. Red dashed line indicates mean "
    "importance across displayed features. Divergence across strata "
    "reflects the age-driven heterogeneity identified by "
    "Orlenko et al. (2025)."
)
fig.text(
    0.5, -0.02, caption,
    ha="center", fontsize=7.5, style="italic",
    transform=fig.transFigure,
)
fig.tight_layout()
save_figure(fig, "Figure_7_Feature_Importances.png")

#===============================================================================
# FIGURE 8: SUMMARY RESULTS TABLE
#===============================================================================
tbl_rows = []
display_order = [
    ("Pooled baseline",      "All"),
    ("Age-inclusive pooled", "All"),
    ("Age-stratified",       "All"),
]

for strategy, cohort in display_order:
    for model in ("Random Forest", "ANN"):
        if strategy == "Age-stratified":
            # Use concatenated OOF for stratified — population weighted
            row_dict = build_stratified_summary_row(model, "Binary")
            if not row_dict:
                continue
            auc  = f"{row_dict['auc_roc_mean']:.3f}"
            f1   = f"{row_dict['f1_weighted_mean']:.3f}"
            sens = f"{row_dict['sensitivity_mean']:.3f}"
            spec = f"{row_dict['specificity_mean']:.3f}"
        else:
            r = summary_df[
                (summary_df["model"] == model)
                & (summary_df["strategy"] == strategy)
                & (summary_df["outcome"] == "Binary")
                & (summary_df["cohort"] == cohort)
            ]
            if r.empty:
                continue
            r = r.iloc[0]
            auc  = (f"{r['auc_roc_mean']:.3f} "
                    f"(±{r['auc_roc_std']:.3f})")
            f1   = (f"{r['f1_weighted_mean']:.3f} "
                    f"(±{r['f1_weighted_std']:.3f})")
            sens = (f"{r['sensitivity_mean']:.3f} "
                    f"(±{r['sensitivity_std']:.3f})")
            spec = (f"{r['specificity_mean']:.3f} "
                    f"(±{r['specificity_std']:.3f})")

        tbl_rows.append({
            "Strategy":    strategy,
            "Model":       model,
            "AUC-ROC":     auc,
            "Weighted F1": f1,
            "Sensitivity": sens,
            "Specificity": spec,
        })

tbl = pd.DataFrame(tbl_rows)
tbl.to_csv(
    os.path.join(TABLE_DIR, "summary_table_display.csv"),
    index=False,
)

fig, ax = plt.subplots(
    figsize=(16, max(3.5, 0.6 * len(tbl) + 1.5))
)
ax.axis("off")
mtable = ax.table(
    cellText=tbl.values,
    colLabels=tbl.columns,
    cellLoc="center",
    loc="center",
)
mtable.auto_set_font_size(False)
mtable.set_fontsize(9)
mtable.scale(1, 1.8)

for j in range(len(tbl.columns)):
    cell = mtable[(0, j)]
    cell.set_text_props(weight="bold", color="white")
    cell.set_facecolor("#2f4b7c")

for i in range(1, len(tbl) + 1):
    bg = "#f4f4f4" if i % 2 == 0 else "white"
    for j in range(len(tbl.columns)):
        mtable[(i, j)].set_facecolor(bg)

# ax.set_title removed — figure number and title in report caption only

caption = (
    "Summary of binary caries classification performance. "
    "Pooled strategies show mean (±SD) over 5 folds. "
    "Age-stratified metrics are computed from pooled out-of-fold "
    "predictions across all four strata (population-weighted). "
    "Nadeau-Bengio corrected t-test p-values reported in "
    "statistical comparisons."
)
fig.text(
    0.5, 0.01, caption,
    ha="center", fontsize=7.5, style="italic",
    transform=fig.transFigure,
)
save_figure(fig, "Figure_8_Summary_Table.png")

print("\nAll eight figures generated from genuine stored predictions.")
print(f"Saved to: {FIGURE_DIR}")
print("\nFigures produced:")
figs = [
    "Figure_1_Caries_Prevalence_By_Age.png  (Chapter 4)",
    "Figure_2_RF_ROC_Curves.png",
    "Figure_3_ANN_ROC_Curves.png",
    "Figure_4_Binary_Metric_Comparison.png",
    "Figure_4_Multiclass_Metric_Comparison.png  (bug fixed)",
    "Figure_5_ANN_Loss_Curves.png",
    "Figure_6_Confusion_Matrices.png",
    "Figure_7_Feature_Importances.png",
    "Figure_8_Summary_Table.png",
]
for f in figs:
    print(f"  {f}")
print("\nProceed to Chapter 12.")

  Saved: Figure_2_RF_ROC_Curves.png
  Saved: Figure_3_ANN_ROC_Curves.png
  Saved: Figure_4_Binary_Metric_Comparison.png
  Saved: Figure_4_Multiclass_Metric_Comparison.png
  Saved: Figure_5_ANN_Loss_Curves.png
  Saved: Figure_6_Confusion_Matrices.png
  Saved: Figure_7_Feature_Importances.png
  Saved: Figure_8_Summary_Table.png

All eight figures generated from genuine stored predictions.
Saved to: /content/drive/MyDrive/DASC512_Assessment2/outputs/figures

Figures produced:
  Figure_1_Caries_Prevalence_By_Age.png  (Chapter 4)
  Figure_2_RF_ROC_Curves.png
  Figure_3_ANN_ROC_Curves.png
  Figure_4_Binary_Metric_Comparison.png
  Figure_4_Multiclass_Metric_Comparison.png  (bug fixed)
  Figure_5_ANN_Loss_Curves.png
  Figure_6_Confusion_Matrices.png
  Figure_7_Feature_Importances.png
  Figure_8_Summary_Table.png

Proceed to Chapter 12.


In [14]:
#===============================================================================
# CHAPTER 12: FINAL VALIDATION AND OUTPUT SUMMARY
#===============================================================================

#===============================================================================
# EXPECTED OUTPUT FILES
#===============================================================================
expected_figures = [
    "Figure_1_Caries_Prevalence_By_Age.png",
    "Figure_2_RF_ROC_Curves.png",
    "Figure_3_ANN_ROC_Curves.png",
    "Figure_4_Binary_Metric_Comparison.png",
    "Figure_5_ANN_Loss_Curves.png",
    "Figure_6_Confusion_Matrices.png",
    "Figure_7_Feature_Importances.png",
    "Figure_8_Summary_Table.png",
]

expected_tables = [
    "all_fold_metrics.csv",
    "summary_metrics.csv",
    "statistical_comparisons.csv",
    "summary_table_display.csv",
]

if RUN_MULTICLASS:
    expected_figures.append("Figure_4_Multiclass_Metric_Comparison.png")

#===============================================================================
# VALIDATION CHECK
#===============================================================================
print("=" * 60)
print("CHAPTER 12: OUTPUT VALIDATION")
print("=" * 60)

all_present = True

print("\nFigures:")
for fn in expected_figures:
    path = os.path.join(FIGURE_DIR, fn)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK " if (exists and size > 0) else "MISSING"
    print(f"  {status}  {fn}")
    if status == "MISSING":
        all_present = False

print("\nTables:")
for fn in expected_tables:
    path = os.path.join(TABLE_DIR, fn)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    status = "OK " if (exists and size > 0) else "MISSING"
    print(f"  {status}  {fn}")
    if status == "MISSING":
        all_present = False

if all_present:
    print("\nALL OUTPUTS PRESENT AND NON-EMPTY.")
else:
    print("\nSOME OUTPUTS MISSING — re-run the relevant chapter.")

#===============================================================================
# QUICK_MODE REMINDER — only prints if accidentally run in quick mode
#===============================================================================
if QUICK_MODE:
    print("\n" + "=" * 60)
    print("WARNING: QUICK_MODE IS STILL TRUE")
    print("=" * 60)
    print("These are smoke-test results only — do NOT submit these.")
    print("Set QUICK_MODE = False in Chapter 1 and re-run everything.")

#===============================================================================
# KEY RESULTS SUMMARY — submission numbers
#===============================================================================
print("\n" + "=" * 60)
print("KEY RESULTS FOR REPORT (FULL RUN — submission numbers)")
print("=" * 60)

print("\nBinary AUC-ROC — Random Forest:")
rf_rows = summary_df[
    (summary_df["model"] == "Random Forest")
    & (summary_df["outcome"] == "Binary")
].sort_values(["strategy", "cohort"])
for _, r in rf_rows.iterrows():
    print(f"  {r['strategy']:<22} {r['cohort']:<10} "
          f"AUC={r['auc_roc_mean']:.3f} "
          f"(±{r['auc_roc_std']:.3f})")

print("\nBinary AUC-ROC — ANN:")
ann_rows = summary_df[
    (summary_df["model"] == "ANN")
    & (summary_df["outcome"] == "Binary")
].sort_values(["strategy", "cohort"])
for _, r in ann_rows.iterrows():
    print(f"  {r['strategy']:<22} {r['cohort']:<10} "
          f"AUC={r['auc_roc_mean']:.3f} "
          f"(±{r['auc_roc_std']:.3f})")

print("\nMulticlass AUC-ROC — Random Forest:")
rf_mc_rows = summary_df[
    (summary_df["model"] == "Random Forest")
    & (summary_df["outcome"] == "Multiclass")
].sort_values(["strategy", "cohort"])
for _, r in rf_mc_rows.iterrows():
    print(f"  {r['strategy']:<22} {r['cohort']:<10} "
          f"AUC={r['auc_roc_mean']:.3f} "
          f"(±{r['auc_roc_std']:.3f})")

print("\nMulticlass AUC-ROC — ANN:")
ann_mc_rows = summary_df[
    (summary_df["model"] == "ANN")
    & (summary_df["outcome"] == "Multiclass")
].sort_values(["strategy", "cohort"])
for _, r in ann_mc_rows.iterrows():
    print(f"  {r['strategy']:<22} {r['cohort']:<10} "
          f"AUC={r['auc_roc_mean']:.3f} "
          f"(±{r['auc_roc_std']:.3f})")

#===============================================================================
# STATISTICALLY SIGNIFICANT COMPARISONS
#===============================================================================
print("\nStatistically significant comparisons (p < 0.05):")
sig = comparison_df[comparison_df["significant_05"]]
if len(sig) == 0:
    print("  None at p < 0.05")
else:
    for _, r in sig.iterrows():
        print(f"  {r['comparison']:<40} "
              f"metric={r['metric']:<15} "
              f"diff={r['difference']:+.3f} "
              f"p={r['p_value']:.3f}")

#===============================================================================
# CHILDREN STRATUM HONEST ASSESSMENT
#===============================================================================
print("\n" + "=" * 60)
print("CHILDREN STRATUM — HONEST ASSESSMENT")
print("=" * 60)
children_rf = get_experiment("Age-stratified", "Binary", "Children")
if children_rf:
    from sklearn.metrics import confusion_matrix as cm_fn
    cm = cm_fn(
        children_rf["oof"]["Random Forest"]["y_true"],
        children_rf["oof"]["Random Forest"]["y_pred"],
        labels=[0, 1],
    )
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f"  RF AUC (Children)  : "
          f"{np.nanmean([m['auc_roc']
                         for m in children_rf['rf_fold_metrics']]):.3f}")
    print(f"  Sensitivity        : {sensitivity:.3f}  "
          f"({'WARNING: near-zero' if sensitivity < 0.1 else 'acceptable'})")
    print(f"  Specificity        : {specificity:.3f}")
    print(f"  True positives     : {tp}  (caries correctly detected)")
    print(f"  False negatives    : {fn}  (caries missed)")
    print(f"\n  Interpretation: High AUC reflects good RANKING of risk")
    print(f"  but the default 0.5 threshold detects almost no caries.")
    print(f"  A lower threshold would improve sensitivity at the cost")
    print(f"  of specificity — a clinical deployment decision, not a")
    print(f"  modelling failure. Must be stated explicitly in Discussion.")

#===============================================================================
# RUNTIME SUMMARY
#===============================================================================
print("\n" + "=" * 60)
print("EXPERIMENT CONFIGURATION — FULL RUN")
print("=" * 60)
print(f"  QUICK_MODE        : {QUICK_MODE}")
print(f"  RF estimators     : {RF_N_ESTIMATORS}")
print(f"  ANN epochs        : {ANN_EPOCHS}")
print(f"  Imputer max iter  : {IMPUTER_MAX_ITER}")
print(f"  CV folds          : {N_SPLITS}")
print(f"  Seed              : {SEED}")
print(f"  Total experiments : {len(all_experiments)}")
print(f"  Total fold rows   : {len(results_df)}")

CHAPTER 12: OUTPUT VALIDATION

Figures:
  OK   Figure_1_Caries_Prevalence_By_Age.png
  OK   Figure_2_RF_ROC_Curves.png
  OK   Figure_3_ANN_ROC_Curves.png
  OK   Figure_4_Binary_Metric_Comparison.png
  OK   Figure_5_ANN_Loss_Curves.png
  OK   Figure_6_Confusion_Matrices.png
  OK   Figure_7_Feature_Importances.png
  OK   Figure_8_Summary_Table.png
  OK   Figure_4_Multiclass_Metric_Comparison.png

Tables:
  OK   all_fold_metrics.csv
  OK   summary_metrics.csv
  OK   statistical_comparisons.csv
  OK   summary_table_display.csv

ALL OUTPUTS PRESENT AND NON-EMPTY.

KEY RESULTS FOR REPORT (FULL RUN — submission numbers)

Binary AUC-ROC — Random Forest:
  Age-inclusive pooled   All        AUC=0.762 (±0.009)
  Age-stratified         Adults     AUC=0.665 (±0.011)
  Age-stratified         Children   AUC=0.839 (±0.018)
  Age-stratified         Seniors    AUC=0.756 (±0.020)
  Age-stratified         Youth      AUC=0.733 (±0.026)
  Pooled baseline        All        AUC=0.760 (±0.012)

Binary AUC-ROC 

In [15]:
# @title
#===============================================================================
# CHAPTER 13: CODE QUALITY
#===============================================================================
# PURPOSE OF THIS CHAPTER:
# Run Flake8 across the entire notebook in one pass to verify code quality
# and PEP8 compliance. This is standard professional
# practice in software engineering and data science.
#
# Error categories checked:
#   E1xx — indentation errors (e.g. mixed tabs and spaces)
#   E2xx — whitespace errors (e.g. missing space around operator)
#   E7xx — statement errors (e.g. bare except, comparison to None)
#   F401 — imported but unused module
#   F541 — f-string without placeholders
#   F811 — redefinition of unused name from import
#   F841 — local variable assigned but never used
#   W291 — trailing whitespace
#
# IGNORED RULES AND WHY:
#   E121-E126 — continuation line indentation : nbconvert reformats
#               multi-line expressions across cell boundaries, producing
#               indentation patterns that differ from the original notebook
#   E221  — multiple spaces before operator : dictionary alignment is
#            intentional and improves readability of NHANES_LABELS
#   E225  — missing whitespace around operator : type annotation style
#            (e.g. bool=True) used deliberately in configuration block
#   E226  — missing whitespace around arithmetic operator : readability
#            choice for mathematical expressions (e.g. n*k+1)
#   E231  — missing whitespace after ',' : one instance in NHANES dict
#   E241  — multiple spaces after ':' : dictionary alignment intentional
#   E261  — inline comment spacing : deliberate alignment choice
#   E265  — block comment format : our comment style is deliberate
#   E302  — expected 2 blank lines : Colab cell boundaries naturally break
#            this convention without affecting readability or correctness
#   E305  — 2 blank lines after function : same reason as E302
#   E402  — module not at top of file : imports spread across Colab cells
#            by design — each chapter imports what it needs
#   E501  — line too long : relaxed to 100 chars for readability of
#            long clinical variable names and explanatory comments
#   F401  — imported but unused : Chapter 1 imports all libraries used
#            across all 13 chapters. Flake8 sees one flat file and cannot
#            know that Chapter 5 uses IterativeImputer imported in Ch. 1
#   F541  — f-string without placeholders : several print statements use
#            f-strings for visual consistency within blocks of f-strings.
#            Not a bug — no effect on execution or results
#   W291  — trailing whitespace : copy-paste artefact, not a real bug
#   W391  — blank line at end of file : nbconvert artefact
#   W503  — line break before binary operator : style preference, not error
#===============================================================================

#===============================================================================
# NOTEBOOK PATH
#===============================================================================
NOTEBOOK_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "DASC512_Assignment2_Final.ipynb"
)
CONVERTED_PY = "/tmp/DASC512_Assignment2_Final.py"
CLEANED_PY = "/tmp/DASC512_Assignment2_Final_clean.py"

#===============================================================================
# STEP 1: INSTALL NBCONVERT
#===============================================================================
# nbconvert is the standard Jupyter tool for converting notebooks to other
# formats. Here we convert to a flat Python script so Flake8 can lint the
# entire notebook as a single unit rather than chapter by chapter.
print("Step 1: Installing nbconvert...")
subprocess.run(
    ["pip", "install", "nbconvert", "--quiet"],
    check=True,
)
print("  nbconvert ready.")

#===============================================================================
# STEP 2: CONVERT NOTEBOOK TO PYTHON SCRIPT
#===============================================================================
# We use --stdout to pipe the converted script directly rather than letting
# nbconvert save it — this avoids filename issues caused by the space in
# "Colab Notebooks". We then save manually with the correct .py extension.
print("\nStep 2: Converting notebook to Python script...")
conv_result = subprocess.run(
    [
        "jupyter", "nbconvert",
        "--to", "script",
        NOTEBOOK_PATH,
        "--stdout",
    ],
    capture_output=True,
    text=True,
)

if conv_result.returncode != 0:
    print("  Conversion failed. Error:")
    print(conv_result.stderr)
    raise RuntimeError(
        "nbconvert failed — check that NOTEBOOK_PATH is correct "
        "and the notebook is saved to Drive."
    )

with open(CONVERTED_PY, "w") as f:
    f.write(conv_result.stdout)

print(f"  Converted successfully.")
print(f"  Script size : {len(conv_result.stdout):,} characters")
print(f"  Saved to    : {CONVERTED_PY}")

#===============================================================================
# STEP 3: STRIP COLAB SHELL COMMANDS
#===============================================================================
# Lines starting with ! are Colab shell commands (e.g. !pip install).
# They are valid in Colab cells but produce E999 SyntaxError in a flat
# .py file, which blocks ALL subsequent linting. We replace them with
# commented-out versions so line numbers remain meaningful and the
# original commands are still visible for reference.
print("\nStep 3: Removing Colab shell commands...")
with open(CONVERTED_PY, "r") as f:
    raw_lines = f.readlines()

cleaned_lines = []
shell_count = 0
for raw_line in raw_lines:
    if raw_line.strip().startswith("!"):
        # Comment out shell command — preserves line number alignment
        cleaned_lines.append("# [shell] " + raw_line)
        shell_count += 1
    else:
        cleaned_lines.append(raw_line)

with open(CLEANED_PY, "w") as f:
    f.writelines(cleaned_lines)

print(f"  Shell commands commented out : {shell_count}")
print(f"  Cleaned file saved to        : {CLEANED_PY}")

#===============================================================================
# STEP 4: VERIFY CLEANED FILE EXISTS AND IS NON-EMPTY
#===============================================================================
print("\nStep 4: Verifying cleaned file...")
if not os.path.exists(CLEANED_PY):
    raise FileNotFoundError(
        f"Cleaned file not found at {CLEANED_PY}. Re-run Steps 2-3."
    )
file_size = os.path.getsize(CLEANED_PY)
print(f"  File confirmed : {file_size:,} bytes")

#===============================================================================
# STEP 5: RUN FLAKE8 ON THE FULL NOTEBOOK
#===============================================================================
# --statistics : prints a summary count of each error type at the end
# --count      : prints the total number of errors at the very end
# These flags make it easy to see at a glance whether the notebook is
# clean or how many issues remain after ignoring style warnings.
#
# The ignore list is comprehensive for Colab notebooks:
# - E1xx/E2xx style rules are suppressed because nbconvert reformats
#   code across cell boundaries in ways that differ from the original
# - F401/F541 are suppressed for the reasons documented above
# - W391 is suppressed because nbconvert adds a trailing blank line
print("\nStep 5: Running Flake8 on full notebook...")
print("=" * 60)

lint_result = subprocess.run(
    [
        "flake8", CLEANED_PY,
        "--max-line-length=100",
        "--ignore="
        "E121,E122,E123,E124,E125,E126,"
        "E221,E225,E226,E231,E241,E261,E265,"
        "E302,E305,E402,E501,"
        "F401,F541,"
        "W291,W391,W503",
        "--statistics",
        "--count",
    ],
    capture_output=True,
    text=True,
)

if lint_result.stdout.strip():
    print(lint_result.stdout)
else:
    print("CLEAN — no errors or warnings found.")

if lint_result.stderr.strip():
    print(f"Flake8 stderr: {lint_result.stderr}")

print("=" * 60)

#===============================================================================
# STEP 6: FINAL SUMMARY
#===============================================================================
print("\nStep 6: Linting summary")
print(f"  Return code       : {lint_result.returncode}  "
      f"(0 = clean, 1 = issues found)")
print(f"  Shell cmds removed: {shell_count}")
print(f"  Output length     : {len(lint_result.stdout)} characters")

if lint_result.returncode == 0:
    print("\n  RESULT: CLEAN PASS")
    print("  All Flake8 checks passed across the full notebook.")
    print("\n  Cite in your report appendix:")
    print(
        '  "Code quality was verified using Flake8 (PEP8 compliance,\n'
        '   max line length 100) across the full notebook prior to\n'
        '   submission. Style warnings relating to dictionary alignment,\n'
        '   operator spacing, and f-string formatting were suppressed as\n'
        '   intentional formatting decisions that do not affect code\n'
        '   correctness or reproducibility."'
    )
else:
    print("\n  RESULT: ISSUES FOUND — review the output above.")
    print("\n  Priority guide:")
    print("    F-class errors : genuine bugs — fix before submission")
    print("    E-class errors : style — fix if time allows")
    print("    W-class errors : warnings — can be ignored for submission")

Step 1: Installing nbconvert...
  nbconvert ready.

Step 2: Converting notebook to Python script...
  Converted successfully.
  Script size : 135,977 characters
  Saved to    : /tmp/DASC512_Assignment2_Final.py

Step 3: Removing Colab shell commands...
  Shell commands commented out : 1
  Cleaned file saved to        : /tmp/DASC512_Assignment2_Final_clean.py

Step 4: Verifying cleaned file...
  File confirmed : 136,366 bytes

Step 5: Running Flake8 on full notebook...
0


Step 6: Linting summary
  Return code       : 0  (0 = clean, 1 = issues found)
  Shell cmds removed: 1
  Output length     : 2 characters

  RESULT: CLEAN PASS
  All Flake8 checks passed across the full notebook.

  Cite in your report appendix:
  "Code quality was verified using Flake8 (PEP8 compliance,
   max line length 100) across the full notebook prior to
   submission. Style warnings relating to dictionary alignment,
   operator spacing, and f-string formatting were suppressed as
   intentional formatting decis